In [ ]:
import numpy as np
import random
from copy import deepcopy

class Semantics:
    def __init__(self, utterances, truth_calc):
        """
        utterances: list of possible utterances (e.g., ["red", "blue"])
        """
        self.utterances = utterances
        self.truth_calc = truth_calc

    def utterance_space(self):
        """
        Return the set of utterances. `n` can be used if space depends on world size.
        """
        return self.utterances

    def is_true(self, utt, obs):
        """
        Returns True/False if utterance is true in a given state.
        """
        return self.truth_calc(utt, obs)

class World:
    def __init__(self, theta, world_parameters, obs_model, obs_generator):
        """
        thetas: list of possible latent states
        obs_model: function (theta, obs) -> likelihood P(obs | theta)
        """
        self.theta = theta
        self.world_parameters = world_parameters
        self.obs_model = obs_model
        self.obs_generator = obs_generator
        self.all_obs = None

    def obs_prob(self, obs, theta):
        """
        Return P(obs | theta).
        """
        return self.obs_model(obs, theta)
    
    def generate_all_obs(self):
        """
        Generate all possible observations using the provided generator function.
        """
        return self.obs_generator(world_parameters=self.world_parameters)
    
    def sample_obs(self):
        """
        Sample an observation according to P(obs | self.theta).
        """
        if self.all_obs is not None:
            all_obs = self.all_obs
        else:
            all_obs = self.generate_all_obs()
            self.all_obs = all_obs
        #print(all_obs)
        probs = np.array([self.obs_prob(obs, self.theta) for obs in all_obs])
        probs = probs / probs.sum()  # normalize to avoid floating-point issues
        return random.choices(all_obs, weights=probs, k=1)[0]

class Belief:
    def __init__(self, values, prior=None):
        """
        values: list of possible joint states, e.g. [(theta, psi), ...]
        prior: optional list of probabilities of same length
        """
        self.values = values
        if prior is None:
            self.prob = np.ones(len(values)) / len(values)
        else:
            self.prob = np.array(prior, dtype=float)
            self.prob /= np.sum(self.prob)

    def update(self, likelihoods):
        """
        likelihoods: array of same length as values
        """
        likelihoods = np.array(likelihoods, dtype=float)
        self.prob *= likelihoods
        if np.sum(self.prob) > 0:
            self.prob /= np.sum(self.prob)

    def as_dict(self):
        return dict(zip(self.values, self.prob))

    def marginal(self, index):
        """
        Compute marginal distribution over one coordinate.
        index = 0 for theta, 1 for psi
        """
        if type(self.values[0]) == float:
            return self.as_dict()
        marg = {}
        for (theta, psi), p in zip(self.values, self.prob):
            key = (theta, psi)[index]
            marg[key] = marg.get(key, 0.0) + p
        return marg

class Speaker0:
    def __init__(self, thetas, semantics, world):
        """
        Speaker stores a belief over theta and reasons using a listener model.
        
        thetas    : list of possible theta values
        listener  : a Listener object from the lower level (e.g. L0 for S1)
        semantics : object defining utterance space, truth conditions
        """
        self.thetas = thetas
        self.belief_theta = Belief(thetas)
        self.semantics = semantics
        self.world = world
        
        self.hist = [deepcopy(self.belief_theta)]
        
        self.utterances_theta = {}

    def infer_state(self, obs):
        """
        Compute posterior over theta given an observation (Bayes update).
        """
        likelihoods = []
        for theta in self.thetas:
            likelihoods.append(self.world.obs_prob(obs, theta))

        posterior = Belief(self.thetas, self.belief_theta.prob.copy())
        posterior.update(likelihoods)
        return posterior

    def update(self, obs):
        """
        Update internal belief over theta after seeing an observation.
        """
        self.belief_theta = self.infer_state(obs)
        self.hist.append(deepcopy(self.belief_theta))
        
        self.utterances_theta = {}  # reset cached utterance distributions
        
        
        return self.belief_theta.as_dict()

    def dist_over_utterances_obs(self, obs):
        """
        For a literal speaker:
        Return uniform distribution over all utterances that are true for the given observation.
        """
        utterances = self.semantics.utterance_space()
        
        # Find all utterances that are true for this observation
        true_utterances = [utt for utt in utterances if self.semantics.is_true(utt, obs)]
        
        # uniform over the true utterances
        p = 1.0 / len(true_utterances)
        probs_dict = {utt: (p if utt in true_utterances else 0.0) for utt in utterances}
        
        return probs_dict

    def dist_over_utterances_theta(self, theta):
        if theta in self.utterances_theta:
            return self.utterances_theta[theta]
        probs = {u: 0.0 for u in self.semantics.utterance_space()}
        for obs in self.world.generate_all_obs():
            obs_prob = self.world.obs_prob(obs, theta)
            for utt, prob in self.dist_over_utterances_obs(obs).items():
                probs[utt] += prob * obs_prob
        self.utterances_theta[theta] = probs
        return probs
    
    def sample_utterance(self, obs):
        """
        Sample an utterance from the utterance distribution.
        """
        dist = self.dist_over_utterances(obs)
        utterances, probs = zip(*dist.items())
        return random.choices(utterances, weights=probs, k=1)[0]

class Listener0:
    def __init__(self, thetas, speaker, world, semantics):
        """
        Listener maintains a joint belief P(theta, psi).
        """
        self.state_belief = Belief(thetas)
        self.speaker = speaker
        self.world = world
        self.semantics = semantics
        
        self.hist = [deepcopy(self.state_belief)]
        
        self.prior_utt = None
        self.obs_utt = {}

    def infer_state(self, utt):
        """
        Compute posterior distribution P(theta, psi | utt) ∝ P(utt | theta, psi) * prior.
        """
        likelihoods = []
        for state in self.state_belief.values:
            # Call speaker model for likelihood
            utt_dist = self.speaker.dist_over_utterances_theta(state)
            likelihoods.append(utt_dist.get(utt, 0.0))
        
        # Posterior
        posterior = Belief(self.state_belief.values, self.state_belief.prob.copy())
        posterior.update(likelihoods)
        return posterior
    
    def infer_obs(self, utt):
        if utt in self.obs_utt:
            return self.obs_utt[utt]
        result = {}
        obs_prob = self.distribution_over_obs()
        utt_priors = self.prior_over_utt()

        for obs in self.world.generate_all_obs():
            # get precomputed priors (cached internally)     
            result[obs] = self.speaker.dist_over_utterances_obs(obs)[utt] * obs_prob[obs] / utt_priors[utt]
        self.obs_utt[utt] = result
        return result

    def distribution_over_obs(self):
        result = {}
        for obs in self.world.generate_all_obs():
            listener_obs = 0
            for (theta, theta_prob) in self.state_belief.as_dict().items():
                listener_obs += self.world.obs_prob(obs, theta) * theta_prob
            result[obs] = listener_obs
        return result
    
    def prior_over_utt(self):
        if self.prior_utt is not None:
            return self.prior_utt
        utt_priors = {}
        for utt in self.semantics.utterance_space():
            total = 0.0
            for obs_case in self.world.generate_all_obs():
                literal_speaker_utterance_obscase_val = self.speaker.dist_over_utterances_obs(obs_case)
                for (theta_case, theta_prob) in self.state_belief.as_dict().items():
                    obs_prior = self.world.obs_prob(obs_case, theta_case)
                    total += (
                        literal_speaker_utterance_obscase_val[utt]
                        * obs_prior
                        * theta_prob
                    )
            utt_priors[utt] = total
        self.prior_utt = utt_priors
        return utt_priors
    
    def update(self, utt):
        """
        Replace the current belief with the posterior after hearing utterance.
        """
        new_belief = self.infer_state(utt)
        self.state_belief = new_belief
        
        self.prior_utt = None
        self.obs_utt = {}
        
        self.hist.append(deepcopy(self.state_belief))
        
        return self.state_belief.as_dict()


class Speaker1:
    def __init__(self, thetas, listener, semantics, world, alpha=1.0, psi="inf"):
        """
        Speaker stores a belief over theta and reasons using a listener model.
        
        thetas    : list of possible theta values
        listener  : a Listener object from the lower level (e.g. L0 for S1)
        semantics : object defining utterance space, truth conditions
        alpha     : rationality parameter
        pers      : persuasion type ("inf", "high", "low")
        """
        self.thetas = thetas
        self.belief_theta = Belief(thetas)
        self.listener = listener
        self.semantics = semantics
        self.world = world
        self.alpha = alpha
        self.psi = psi
        
        self.hist = [deepcopy(self.belief_theta)]
        
        self.utterance_theta_psi = {}
        self.informativeness_obs_utt = {}
        self.persuasiveness_psi = {}
        self.utterances_obs_psi = {}

    def infer_state(self, obs):
        """
        Compute posterior over theta given an observation (Bayes update).
        """
        likelihoods = []
        for theta in self.thetas:
            likelihoods.append(self.world.obs_prob(obs, theta))

        posterior = Belief(self.thetas, self.belief_theta.prob.copy())
        posterior.update(likelihoods)
        return posterior

    def update(self, obs):
        """
        Update internal belief over theta after seeing an observation.
        """
        self.belief_theta = self.infer_state(obs)
        self.hist.append(deepcopy(self.belief_theta))
        
        self.utterance_theta_psi = {}
        self.informativeness_obs_utt = {}
        self.persuasiveness_psi = {}
        self.utterances_obs_psi = {}

        return self.belief_theta.as_dict()

    def get_informativeness_obs_utt(self, obs, utt):
        if (obs, utt) in self.informativeness_obs_utt:
            return self.informativeness_obs_utt[(obs, utt)]
        speaker_dist = self.infer_state(obs).as_dict()
        listener_dist = self.listener.infer_state(utt).marginal(0)

        result = 0
        for (theta, prob) in speaker_dist.items():
            if listener_dist[theta] == 0:
                if prob == 0:
                    continue
                else:
                    result = float('-inf')
                    break
            else:
                result += np.log2(listener_dist[theta]) * prob
        self.informativeness_obs_utt[(obs, utt)] = result
        return result
    
    def get_informativeness_obs(self, obs):
        result = {u: 0.0 for u in self.semantics.utterance_space()}
        speaker_dist = self.infer_state(obs).as_dict()
        for utt in self.semantics.utterance_space():
            listener_dist = self.listener.infer_state(utt).marginal(0)
            for (theta, prob) in speaker_dist.items():
                if listener_dist[theta] == 0:
                    if prob == 0:
                        continue
                    else:
                        result[utt] = float('-inf')
                        break
                else:
                    result[utt] += np.log2(listener_dist[theta]) * prob
        return result
    
    # def get_informativeness_obs_utt(self, obs, utt):
    #     if (obs, utt) in self.informativeness_obs_utt:
    #         return self.informativeness_obs_utt[(obs, utt)]
    #     result = self.listener.infer_obs(utt)
    #     for (obs_case, prob) in result.items():
    #         self.informativeness_obs_utt[(obs_case, utt)] = prob
    #     return result[obs]
    
    # def get_informativeness_obs(self, obs):
    #     result = {}
    #     for utt in self.semantics.utterance_space():
    #         result[utt] = self.listener.infer_obs(utt)[obs]
    #     return result

    # def get_persuasiveness(self, pers, obs=None):
    #     if pers in self.persuasiveness_psi:
    #         return self.persuasiveness_psi[pers]
    #     utterances = self.semantics.utterance_space()
    #     result = {u: 0.0 for u in utterances}

    #     for utt in utterances:
    #         if pers == "inf":
    #             result[utt] = 1
    #         elif pers == "high":
    #             for (theta, theta_prob) in self.listener.infer_state(utt).as_dict().items():
    #                 result[utt] += theta * theta_prob
    #         elif pers == "low":
    #             for (theta, theta_prob) in self.listener.infer_state(utt).as_dict().items():
    #                 result[utt] += theta * theta_prob
    #             result[utt] = 1 - result[utt]
    #     self.persuasiveness_psi[pers] = result
    #     return result

    def get_persuasiveness(self, pers, obs, debug = False):
        if (pers, obs) in self.persuasiveness_psi:
            return self.persuasiveness_psi[(pers, obs)]
        utterances = self.semantics.utterance_space()
        result = {u: 0.0 for u in utterances}

        current_listener_belief = self.listener.state_belief.marginal(0)
        current_listener_mean = 0
        for (theta, prob) in current_listener_belief.items():
            current_listener_mean += theta * prob
        
        pers_mean = 0
        amount = 0
        
        if debug:
            print("Current listener mean:", current_listener_mean)
        max_pers = float('-inf')
        min_pers = float('inf')
        for utt in utterances:
            if pers == "inf":
                result[utt] = 1
            elif pers == "high":
                for (theta, state_prob) in self.listener.infer_state(utt).marginal(0).items():
                    result[utt] += theta * state_prob
            elif pers == "low":
                for (theta, state_prob) in self.listener.infer_state(utt).marginal(0).items():
                    result[utt] += theta * state_prob
                #result[utt] = current_listener_mean / result[utt]
                result[utt] = 1 - result[utt]

            if debug:
                print(f"{utt}: {result[utt]}")
            if self.semantics.is_true(utt, obs):
                pers_mean += result[utt]
                amount += 1
            if result[utt] > max_pers and self.semantics.is_true(utt, obs):
                    max_pers = result[utt]
            if result[utt] < min_pers and self.semantics.is_true(utt, obs):
                    min_pers = result[utt]
        
        pers_mean /= amount
        #print("Persuasiveness mean over true utts:", pers_mean)
        for utt in utterances:
            if pers == "high":
                result[utt] = result[utt] - pers_mean
            elif pers == "low":
                result[utt] = pers_mean - result[utt]


        self.persuasiveness_psi[(pers, obs)] = result
        return result

    def dist_over_utterances_obs(self, obs, psi):
        """
        Distribution over utterances given observation.
        Uses informativeness + persuasiveness.
        """
        if (obs, psi) in self.utterances_obs_psi:
            return self.utterances_obs_psi[(obs, psi)]
        utterances = self.semantics.utterance_space()
        persuasiveness = self.get_persuasiveness(psi, obs)
        scores = []
        if psi == "inf":
            beta = 1.0
        else:
            beta = 0.0

        true_utterances = [utt for utt in utterances if self.semantics.is_true(utt, obs)]

        for utt in utterances:
            info_val = self.get_informativeness_obs_utt(obs, utt)
            pers_val = persuasiveness[utt]
            if self.semantics.is_true(utt, obs):
                #score = (info_val ** (self.alpha * beta)) * (pers_val ** (self.alpha * (1 - beta)))
                if psi == "inf":
                    score = np.exp(self.alpha * info_val)
                else:
                    if pers_val <= 0:
                        score = 0.0
                    else:
                        score = np.exp(self.alpha * np.log2(pers_val) * (1 - beta))
            else:
                score = 0.0
            scores.append(score)

        scores = np.array(scores)
        if np.sum(scores) == 0:
            for i, utt in enumerate(utterances):
                if utt in true_utterances:
                    scores[i] = 1.0

        probs = scores / np.sum(scores)
        self.utterances_obs_psi[(obs, psi)] = dict(zip(utterances, probs))
        return self.utterances_obs_psi[(obs, psi)]

    def dist_over_utterances_theta(self, theta, psi):
        if (theta, psi) in self.utterance_theta_psi:
            return self.utterance_theta_psi[(theta, psi)]
        result = {u: 0.0 for u in self.semantics.utterance_space()}
        for obs in self.world.generate_all_obs():
            obs_prob = self.world.obs_prob(obs, theta)
            for utt, prob in self.dist_over_utterances_obs(obs, psi).items():
                result[utt] += prob * obs_prob
        self.utterance_theta_psi[(theta, psi)] = result
        return result
    
    def sample_utterance(self, obs):
        """
        Sample an utterance from the utterance distribution.
        """
        dist = self.dist_over_utterances_obs(obs, self.psi)
        utterances, probs = zip(*dist.items())
        return random.choices(utterances, weights=probs, k=1)[0]


class Listener1:
    def __init__(self, thetas, psis, speaker, world, semantics, listener_type, threshold = 0.6, alpha=1.0):
        """
        Listener maintains a joint belief P(theta, psi).
        """
        if listener_type == "inf":
            psis = ["inf"]
        joint_values = [(theta, psi) for theta in thetas for psi in psis]
        self.state_belief = Belief(joint_values)
        self.speaker = speaker
        self.world = world
        self.semantics = semantics
        self.psis = psis
        self.alpha = alpha
        self.threshold = threshold
        self.listener_type = listener_type
        self.thetas = thetas
        
        self.hist = [deepcopy(self.state_belief)]
        self.suspicion = []
        self.utt_history = []
        
        self.obs_psi_utt = {}
        self.obs_psi = None
        self.prior_utt = None
        self.obs_utt = {}
        self.suspicions = {}

    def infer_state(self, utt):
        """
        Compute posterior distribution P(theta, psi | utt) ∝ P(utt | theta, psi) * prior.
        """
            
        likelihoods = []
        for state in self.state_belief.values:
            # Call speaker model for likelihood
            utt_dist = self.speaker.dist_over_utterances_theta(state[0], state[1])
            likelihoods.append(utt_dist.get(utt, 0.0))
        
        # Posterior
        posterior = Belief(self.state_belief.values, self.state_belief.prob.copy())
        posterior.update(likelihoods)
        return posterior
                    
    def infer_obs(self, utt):
        
        if utt in self.obs_utt:
            return self.obs_utt[utt]
        result = {}
        obs_psi_utt = self.infer_obs_psi(utt)
        psi_probs = self.marginal_psi()
        for obs in self.world.generate_all_obs():
            for psi, prob in psi_probs.items():
                result[obs] = result.get(obs, 0.0) + obs_psi_utt[(obs, psi)]
        self.obs_utt[utt] = result
        return result
    
    def infer_obs_psi(self, utt):
        if utt in self.obs_psi_utt:
            return self.obs_psi_utt[utt]
        result = {(obs,psi): 0.0 for obs in self.world.generate_all_obs() for psi in self.psis}
        obs_psi= self.distribution_over_obs_psi()
        utt_priors = self.prior_over_utt()
        for obs in self.world.generate_all_obs():
            # get precomputed priors (cached internally)   
            for psi in self.psis:
                result[(obs, psi)] = self.speaker.dist_over_utterances_obs(obs, psi)[utt] * obs_psi[(obs, psi)] / utt_priors[utt]
        self.obs_psi_utt[utt] = result
        return result
    
    def distribution_over_obs_psi(self):
        if self.obs_psi is not None:
            return self.obs_psi
        
        result = {(obs,psi): 0.0 for obs in self.world.generate_all_obs() for psi in self.psis}
        for (state, state_prob) in self.state_belief.as_dict().items():
            for obs in self.world.generate_all_obs():
                cur_obs_prob = self.world.obs_prob(obs, state[0])
                result[(obs, state[1])] += cur_obs_prob * state_prob
        self.obs_psi_utt = result
        return result
    
    def prior_over_utt(self):
        if self.prior_utt is not None:
            return self.prior_utt
        utt_priors = {utt: 0.0 for utt in self.semantics.utterance_space()}
        obs_psi_dist = self.distribution_over_obs_psi()
        for obs_case in self.world.generate_all_obs():
            for psi in self.psis:
                pragmatic_speaker_utterance_obscase_val = self.speaker.dist_over_utterances_obs(obs_case, psi)
                for utt in self.semantics.utterance_space():
                    utt_priors[utt] += pragmatic_speaker_utterance_obscase_val[utt] * obs_psi_dist[(obs_case, psi)]
                    
        self.prior_utt = utt_priors
        return utt_priors
    
    def update(self, utt, recomputing = False):
        """
        Replace the current belief with the posterior after hearing utterance.
        """

        self.suspicion.append(self.get_suspicion(utt))
        self.utt_history.append(utt)
        new_belief = self.infer_state(utt)
        self.state_belief = new_belief

        self.hist.append(deepcopy(self.state_belief))
        self.obs_psi_utt = {}
        self.obs_psi = None
        self.prior_utt = None
        self.obs_utt = {}
        
        return self.state_belief
    
    def marginal_theta(self):
        """
        Return the marginal distribution over theta.
        """
        theta_probs = {}
        for (theta, psi), p in zip(self.state_belief.values, self.state_belief.prob):
            theta_probs[theta] = theta_probs.get(theta, 0.0) + p
        return theta_probs

    def marginal_psi(self):
        """
        Return the marginal distribution over psi.
        """
        psi_probs = {}
        for (theta, psi), p in zip(self.state_belief.values, self.state_belief.prob):
            psi_probs[psi] = psi_probs.get(psi, 0.0) + p
        return psi_probs
    
    def get_suspicion(self, utt):
        """
        Compute the suspicion of an utterance.
        """
        if utt in self.suspicions:
            return self.suspicions[utt]
        
        suspicion = 0.0
        for state, state_prior in self.speaker.listener.infer_state(utt).marginal(0).items():
            speaker_probs = self.speaker.dist_over_utterances_theta(state, "inf")
            for u, prob in speaker_probs.items():
                if np.isclose(prob, speaker_probs[utt], rtol=1e-9, atol=1e-12):
                    continue
                elif prob > speaker_probs[utt]:
                    suspicion += state_prior * prob
        self.suspicions[utt] = suspicion
        return suspicion
    
    def false_positive_rates(self, thresholds):
        fprs = {threshold: 0.0 for threshold in thresholds}
        for state, prob in self.state_belief.marginal(0).items():
            speaker_dist = self.speaker.dist_over_utterances_theta(state, "inf")
            for u in self.semantics.utterance_space():
                for threshold in thresholds:
                    if self.get_suspicion(u) >= threshold:
                        fprs[threshold] += speaker_dist[u] * prob
        return fprs

    def true_positive_rates(self, thresholds, pers_ratio = 0.5):
        tprs = {threshold: 0.0 for threshold in thresholds}
        for state, prob in self.state_belief.marginal(0).items():
            speaker_dist_high = self.speaker.dist_over_utterances_theta(state, "high")
            speaker_dist_low = self.speaker.dist_over_utterances_theta(state, "low")
            for u in self.semantics.utterance_space():
                for threshold in thresholds:
                    if self.get_suspicion(u) >= threshold:
                        tprs[threshold] += speaker_dist_high[u] * prob * pers_ratio
                        tprs[threshold] += speaker_dist_low[u] * prob * (1 - pers_ratio)
        return tprs

class Listener1Switch:
    def __init__(self, thetas, psis, speaker, world, semantics, threshold = 0.6, alpha=1.0):
        """
        Listener maintains a joint belief P(theta, psi).
        """
        self.psis = ["inf"]
        self.thetas = thetas
        joint_values = [(theta, psi) for theta in self.thetas for psi in self.psis]
        self.state_belief = Belief(joint_values)
        self.speaker = speaker
        self.world = world
        self.semantics = semantics
        self.alpha = alpha
        self.threshold = threshold
        self.switched = False
        self.switch_point = 1

        self.vig_listener = Listener1(thetas, psis, speaker, world, semantics, listener_type="vig", alpha=alpha)
        self.vig_listener.state_belief = deepcopy(self.state_belief)
        self.inf_listener = Listener1(thetas, psis, speaker, world, semantics, listener_type="inf", alpha=alpha)
        
        self.hist = [deepcopy(self.state_belief)]
        self.suspicion = []
        self.utt_history = []
        
        self.obs_psi_utt = {}
        self.obs_psi = None
        self.prior_utt = None
        self.obs_utt = {}
        self.suspicions = {}

    def infer_state(self, utt):
        """
        Compute posterior distribution P(theta, psi | utt) ∝ P(utt | theta, psi) * prior.
        """
        # if type is switch and not yet switched:
        current_sus = self.get_suspicion(utt)
        if self.switched or current_sus >= self.threshold:
            return self.vig_listener.infer_state(utt)
        else:
            return self.inf_listener.infer_state(utt)
                    
    def infer_obs(self, utt):
        current_sus = self.get_suspicion(utt)
        if self.switched or current_sus >= self.threshold:
            return self.vig_listener.infer_obs(utt)
        else:
            return self.inf_listener.infer_obs(utt)

    def infer_obs_psi(self, utt):
        current_sus = self.get_suspicion(utt)
        if self.switched or current_sus >= self.threshold:
            return self.vig_listener.infer_obs_psi(utt)
        else:
            return self.inf_listener.infer_obs_psi(utt)
    
    def distribution_over_obs_psi(self):
        if self.switched:
            return self.vig_listener.distribution_over_obs_psi()
        else:
            return self.inf_listener.distribution_over_obs_psi()

    def prior_over_utt(self):
        if self.switched:
            return self.vig_listener.prior_over_utt()
        else:
            return self.inf_listener.prior_over_utt()

    def update(self, utt):
        """
        Replace the current belief with the posterior after hearing utterance.
        """
        current_sus = self.get_suspicion(utt)
        self.suspicion.append(current_sus)
        vig_belief = self.vig_listener.update(utt)
        inf_belief = self.inf_listener.update(utt)
        
        if current_sus >= self.threshold:
            self.switched = True
            
        if self.switched:
            new_belief = vig_belief
        else:
            new_belief = inf_belief
            self.switch_point += 1

        self.utt_history.append(utt)
        self.state_belief = new_belief

        self.suspicions = {}
        self.hist.append(deepcopy(self.state_belief))
        
        return new_belief
                
                
    def marginal_theta(self):
        """
        Return the marginal distribution over theta.
        """
        if self.switched:
            return self.vig_listener.marginal_theta()
        else:
            return self.inf_listener.marginal_theta()

    def marginal_psi(self):
        """
        Return the marginal distribution over psi.
        """
        if self.switched:
            return self.vig_listener.marginal_psi()
        else:
            return self.inf_listener.marginal_psi()
    
    def get_suspicion(self, utt):
        """
        Compute the suspicion of an utterance.
        """
        if utt in self.suspicions:
            return self.suspicions[utt]
        
        suspicion = 0.0
        for state, state_prior in self.speaker.listener.infer_state(utt).marginal(0).items():
            speaker_probs = self.speaker.dist_over_utterances_theta(state, "inf")
            for u, prob in speaker_probs.items():
                if np.isclose(prob, speaker_probs[utt], rtol=1e-9, atol=1e-12):
                    continue
                elif prob > speaker_probs[utt]:
                    suspicion += state_prior * prob
        self.suspicions[utt] = suspicion
        return suspicion
    
    # def false_positive_rates(self, thresholds):
    #     fprs = {threshold: 0.0 for threshold in thresholds}
    #     for state, prob in self.state_belief.marginal(0).items():
    #         speaker_dist = self.speaker.dist_over_utterances_theta(state, "inf")
    #         for u in self.semantics.utterance_space():
    #             for threshold in thresholds:
    #                 if self.get_suspicion(u) >= threshold:
    #                     fprs[threshold] += speaker_dist[u] * prob
    #     return fprs

    # def true_positive_rates(self, thresholds, pers_ratio = 0.5):
    #     tprs = {threshold: 0.0 for threshold in thresholds}
    #     for state, prob in self.state_belief.marginal(0).items():
    #         speaker_dist_high = self.speaker.dist_over_utterances_theta(state, "high")
    #         speaker_dist_low = self.speaker.dist_over_utterances_theta(state, "low")
    #         for u in self.semantics.utterance_space():
    #             for threshold in thresholds:
    #                 if self.get_suspicion(u) >= threshold:
    #                     tprs[threshold] += speaker_dist_high[u] * prob * pers_ratio
    #                     tprs[threshold] += speaker_dist_low[u] * prob * (1 - pers_ratio)
    
    def false_positive_rates(self, thresholds):
        fprs = {threshold: 0.0 for threshold in thresholds}
        for obs in self.world.generate_all_obs():
            speaker_dist = self.speaker.dist_over_utterances_obs(obs, "inf")
            prob = self.world.obs_prob(obs, self.world.theta)
            for u in self.semantics.utterance_space():
                for threshold in thresholds:
                    if self.get_suspicion(u) >= threshold:
                        fprs[threshold] += speaker_dist[u] * prob
        return fprs

    def true_positive_rates(self, thresholds, pers_ratio = 0.5):
        tprs = {threshold: 0.0 for threshold in thresholds}
        for obs in self.world.generate_all_obs():
            speaker_dist_high = self.speaker.dist_over_utterances_obs(obs, "high")
            speaker_dist_low = self.speaker.dist_over_utterances_obs(obs, "low")
            prob = self.world.obs_prob(obs, self.world.theta)
            for u in self.semantics.utterance_space():
                for threshold in thresholds:
                    if self.get_suspicion(u) >= threshold:
                        tprs[threshold] += speaker_dist_high[u] * prob * pers_ratio
                        tprs[threshold] += speaker_dist_low[u] * prob * (1 - pers_ratio)
        return tprs

class Speaker2:
    def __init__(self, thetas, listener, semantics, world, alpha=1.0, psi="inf"):
        """
        Speaker stores a belief over theta and reasons using a listener model.
        
        thetas    : list of possible theta values
        listener  : a Listener object from the lower level (e.g. L0 for S1)
        semantics : object defining utterance space, truth conditions
        alpha     : rationality parameter
        pers      : persuasion type ("inf", "high", "low")
        """
        self.thetas = thetas
        self.belief_theta = Belief(thetas)
        self.listener = listener
        self.semantics = semantics
        self.world = world
        self.alpha = alpha
        self.psi = psi
        
        self.hist = [deepcopy(self.belief_theta)]
        self.prob_hist = []
        
        self.utterance_theta_psi = {}
        self.informativeness_obs_utt = {}
        self.persuasiveness_psi = {}
        self.utterances_obs_psi = {}

    def infer_state(self, obs):
        """
        Compute posterior over theta given an observation (Bayes update).
        """
        likelihoods = []
        for theta in self.thetas:
            likelihoods.append(self.world.obs_prob(obs, theta))

        posterior = Belief(self.thetas, self.belief_theta.prob.copy())
        posterior.update(likelihoods)
        return posterior

    def update(self, obs):
        """
        Update internal belief over theta after seeing an observation.
        """
        self.belief_theta = self.infer_state(obs)
        self.hist.append(deepcopy(self.belief_theta))
        
        self.utterance_theta_psi = {}
        self.informativeness_obs_utt = {}
        self.persuasiveness_psi = {}
        self.utterances_obs_psi = {}
        return self.belief_theta.as_dict()

    # def get_informativeness_obs_utt(self, obs, utt):
    #     if (obs, utt) in self.informativeness_obs_utt:
    #         return self.informativeness_obs_utt[(obs, utt)]
    #     result = self.listener.infer_obs(utt)
    #     for (obs_case, prob) in result.items():
    #         self.informativeness_obs_utt[(obs_case, utt)] = prob
    #     return result[obs]
    
    # def get_informativeness_obs(self, obs):
    #     result = {}
    #     for utt in self.semantics.utterance_space():
    #         result[utt] = self.listener.infer_obs(utt)[obs]
    #     return result
    
    def get_informativeness_obs_utt(self, obs, utt):
        if (obs, utt) in self.informativeness_obs_utt:
            return self.informativeness_obs_utt[(obs, utt)]
        speaker_dist = self.infer_state(obs).as_dict()
        listener_dist = self.listener.infer_state(utt).marginal(0)
        result = 0
        for (theta, prob) in speaker_dist.items():
            if listener_dist[theta] == 0:
                if prob == 0:
                    continue
                else:
                    result = float('-inf')
                    break
            else:
                result += np.log2(listener_dist[theta]) * prob
        self.informativeness_obs_utt[(obs, utt)] = result
        return result
    
    def get_informativeness_obs(self, obs):
        result = {u: 0.0 for u in self.semantics.utterance_space()}
        speaker_dist = self.infer_state(obs).as_dict()
        for utt in self.semantics.utterance_space():
            listener_dist = self.listener.infer_state(utt).marginal(0)
            for (theta, prob) in speaker_dist.items():
                if listener_dist[theta] == 0:
                    if prob == 0:
                        continue
                    else:
                        result[utt] = float('-inf')
                        break
                else:
                    result[utt] += np.log2(listener_dist[theta]) * prob
        return result

    # def get_persuasiveness(self, pers, obs = None):
    #     if pers in self.persuasiveness_psi:
    #         return self.persuasiveness_psi[pers]
    #     utterances = self.semantics.utterance_space()
    #     result = {u: 0.0 for u in utterances}

    #     for utt in utterances:
    #         if pers == "inf":
    #             result[utt] = 1
    #         elif pers == "high":
    #             for (theta, theta_prob) in self.listener.infer_state(utt).marginal(0).items():
    #                 result[utt] += theta * theta_prob
    #         elif pers == "low":
    #             for (theta, theta_prob) in self.listener.infer_state(utt).marginal(0).items():
    #                 result[utt] += theta * theta_prob
    #             result[utt] = 1 - result[utt]
    #     self.persuasiveness_psi[pers] = result
    #     return result

    def get_persuasiveness(self, pers, obs, debugger = False):
        if (pers, obs) in self.persuasiveness_psi:
            return self.persuasiveness_psi[(pers, obs)]
        utterances = self.semantics.utterance_space()
        result = {u: 0.0 for u in utterances}
        current_listener_belief = self.listener.state_belief.marginal(0)
        #print("asdf", current_listener_belief)
        current_listener_mean = 0
        for (theta, prob) in current_listener_belief.items():
            current_listener_mean += theta * prob

        pers_mean = 0
        amount = 0

        #print("gotunu sikeyim")
        if debugger:
            print("current listener mean:", current_listener_mean)
        
        max_pers = float('-inf')
        min_pers = float('inf')
        for utt in utterances:
            if pers == "inf":
                result[utt] = 1
            elif pers == "high":
                for (state, state_prob) in self.listener.infer_state(utt).as_dict().items():
                    result[utt] += state[0] * state_prob
            elif pers == "low":
                for (state, state_prob) in self.listener.infer_state(utt).as_dict().items():
                    result[utt] += state[0] * state_prob
                #result[utt] = current_listener_mean / result[utt]
                result[utt] = 1 - result[utt]
            
            if result[utt] > max_pers and self.semantics.is_true(utt, obs):
                    max_pers = result[utt]
            if result[utt] < min_pers and self.semantics.is_true(utt, obs):
                    min_pers = result[utt]
            if debugger:
                print(f"{utt}: {result[utt]}")
            if self.semantics.is_true(utt, obs):
                pers_mean += result[utt]
                amount += 1
        
        pers_mean /= amount
        for utt in utterances:
            if pers == "high":
                result[utt] = result[utt] - pers_mean
            elif pers == "low":
                result[utt] = pers_mean - result[utt]

        self.persuasiveness_psi[(pers, obs)] = result
        return result

    def dist_over_utterances_obs(self, obs, psi):
        """
        Distribution over utterances given observation.
        Uses informativeness + persuasiveness.
        """
        if (obs, psi) in self.utterances_obs_psi:
            return self.utterances_obs_psi[(obs, psi)]
        utterances = self.semantics.utterance_space()
        persuasiveness = self.get_persuasiveness(psi, obs)
        true_utterances = [utt for utt in utterances if self.semantics.is_true(utt, obs)]
        scores = []
        if psi == "inf":
            beta = 1.0
        else:
            beta = 0.0
        
        for utt in utterances:
            info_val = self.get_informativeness_obs_utt(obs, utt)
            pers_val = persuasiveness[utt]
            if self.semantics.is_true(utt, obs):
                #score = (info_val ** (self.alpha * beta)) * (pers_val ** (self.alpha * (1 - beta)))
                if psi == "inf":
                    score = np.exp(self.alpha * info_val)
                else:
                    if pers_val <= 0:
                        score = 0.0
                    else:
                        score = np.exp(self.alpha * np.log2(pers_val) * (1 - beta))
            else:
                score = 0.0
            scores.append(score)
        #print("scores",scores)
        scores = np.array(scores)
        if np.sum(scores) == 0:
            for i, utt in enumerate(utterances):
                if utt in true_utterances:
                    scores[i] = 1.0
        
        # if np.sum(scores) == 0:
        #     print(f"obs: {obs}, psi: {psi}")
        #     print("pers:", persuasiveness)
        #     print("scores:", scores)
        probs = scores / np.sum(scores)
        self.utterances_obs_psi[(obs, psi)] = dict(zip(utterances, probs))
        return self.utterances_obs_psi[(obs, psi)]

    def dist_over_utterances_theta(self, theta, psi):
        if (theta, psi) in self.utterance_theta_psi:
            return self.utterance_theta_psi[(theta, psi)]
        result = {u: 0.0 for u in self.semantics.utterance_space()}
        for obs in self.world.generate_all_obs():
            obs_prob = self.world.obs_prob(obs, theta)
            for utt, prob in self.dist_over_utterances_obs(obs, psi).items():
                result[utt] += prob * obs_prob
        self.utterance_theta_psi[(theta, psi)] = result
        return result
    
    def sample_utterance(self, obs):
        """
        Sample an utterance from the utterance distribution.
        """
        self.prob_hist.append(self.dist_over_utterances_obs((0,0,0,0,1,0,0,0), self.psi))
        dist = self.dist_over_utterances_obs(obs, self.psi)
        # print(dist)
        utterances, probs = zip(*dist.items())
        return random.choices(utterances, weights=probs, k=1)[0]

class Listener2Switch:
    def __init__(self, thetas, psis, speaker, world, semantics, threshold = 0.6, alpha=1.0):
        """
        Listener maintains a joint belief P(theta, psi).
        """
        self.psis = ["inf"]
        self.thetas = thetas
        joint_values = [(theta, psi) for theta in self.thetas for psi in self.psis]
        self.state_belief = Belief(joint_values)
        self.speaker = speaker
        self.world = world
        self.semantics = semantics
        self.alpha = alpha
        self.threshold = threshold
        self.switched = False
        self.switch_point = 1

        self.vig_listener = Listener1Switch(thetas, psis, speaker, world, semantics, alpha=alpha)
        self.vig_listener.state_belief = deepcopy(self.state_belief)
        self.inf_listener = Listener1(thetas, psis, speaker, world, semantics, listener_type="inf", alpha=alpha)
        
        self.hist = [deepcopy(self.state_belief)]
        self.suspicion = []
        self.utt_history = []
        
        self.obs_psi_utt = {}
        self.obs_psi = None
        self.prior_utt = None
        self.obs_utt = {}
        self.suspicions = {}


    def infer_state(self, utt):
        """
        Compute posterior distribution P(theta, psi | utt) ∝ P(utt | theta, psi) * prior.
        """
        # if type is switch and not yet switched:
        current_sus = self.get_suspicion(utt)
        if self.switched or current_sus >= self.threshold:
            return self.vig_listener.infer_state(utt)
        else:
            return self.inf_listener.infer_state(utt)
                    
    def infer_obs(self, utt):
        current_sus = self.get_suspicion(utt)
        if self.switched or current_sus >= self.threshold:
            return self.vig_listener.infer_obs(utt)
        else:
            return self.inf_listener.infer_obs(utt)

    def infer_obs_psi(self, utt):
        current_sus = self.get_suspicion(utt)
        if self.switched or current_sus >= self.threshold:
            return self.vig_listener.infer_obs_psi(utt)
        else:
            return self.inf_listener.infer_obs_psi(utt)
    
    def distribution_over_obs_psi(self):
        if self.switched:
            return self.vig_listener.distribution_over_obs_psi()
        else:
            return self.inf_listener.distribution_over_obs_psi()

    def prior_over_utt(self):
        if self.switched:
            return self.vig_listener.prior_over_utt()
        else:
            return self.inf_listener.prior_over_utt()

    def update(self, utt):
        """
        Replace the current belief with the posterior after hearing utterance.
        """
        current_sus = self.get_suspicion(utt)
        self.suspicion.append(current_sus)
        vig_belief = self.vig_listener.update(utt)
        inf_belief = self.inf_listener.update(utt)
        
        if current_sus >= self.threshold:
            self.switched = True
            
        if self.switched:
            new_belief = vig_belief
        else:
            new_belief = inf_belief
            self.switch_point += 1

        self.utt_history.append(utt)
        self.state_belief = new_belief

        self.suspicions = {}
        self.hist.append(deepcopy(self.state_belief))
        
        return new_belief
                
                
    def marginal_theta(self):
        """
        Return the marginal distribution over theta.
        """
        if self.switched:
            return self.vig_listener.marginal_theta()
        else:
            return self.inf_listener.marginal_theta()

    def marginal_psi(self):
        """
        Return the marginal distribution over psi.
        """
        if self.switched:
            return self.vig_listener.marginal_psi()
        else:
            return self.inf_listener.marginal_psi()
    
    def get_suspicion(self, utt):
        """
        Compute the suspicion of an utterance.
        """
        if utt in self.suspicions:
            return self.suspicions[utt]
        
        suspicion = 0.0
        for state, state_prior in self.speaker.listener.infer_state(utt).marginal(0).items():
            speaker_probs = self.speaker.dist_over_utterances_theta(state, "inf")
            for u, prob in speaker_probs.items():
                if np.isclose(prob, speaker_probs[utt], rtol=1e-9, atol=1e-12):
                    continue
                elif prob > speaker_probs[utt]:
                    suspicion += state_prior * prob
        self.suspicions[utt] = suspicion
        return suspicion
    
    # def false_positive_rates(self, thresholds):
    #     fprs = {threshold: 0.0 for threshold in thresholds}
    #     for state, prob in self.state_belief.marginal(0).items():
    #         speaker_dist = self.speaker.dist_over_utterances_theta(state, "inf")
    #         for u in self.semantics.utterance_space():
    #             for threshold in thresholds:
    #                 if self.get_suspicion(u) >= threshold:
    #                     fprs[threshold] += speaker_dist[u] * prob
    #     return fprs

    # def true_positive_rates(self, thresholds, pers_ratio = 0.5):
    #     tprs = {threshold: 0.0 for threshold in thresholds}
    #     for state, prob in self.state_belief.marginal(0).items():
    #         speaker_dist_high = self.speaker.dist_over_utterances_theta(state, "high")
    #         speaker_dist_low = self.speaker.dist_over_utterances_theta(state, "low")
    #         for u in self.semantics.utterance_space():
    #             for threshold in thresholds:
    #                 if self.get_suspicion(u) >= threshold:
    #                     tprs[threshold] += speaker_dist_high[u] * prob * pers_ratio
    #                     tprs[threshold] += speaker_dist_low[u] * prob * (1 - pers_ratio)
    
    def false_positive_rates(self, thresholds):
        fprs = {threshold: 0.0 for threshold in thresholds}
        for obs in self.world.generate_all_obs():
            speaker_dist = self.speaker.dist_over_utterances_obs(obs, "inf")
            prob = self.world.obs_prob(obs, self.world.theta)
            for u in self.semantics.utterance_space():
                for threshold in thresholds:
                    if self.get_suspicion(u) >= threshold:
                        fprs[threshold] += speaker_dist[u] * prob
        return fprs

    def true_positive_rates(self, thresholds, pers_ratio = 0.5):
        tprs = {threshold: 0.0 for threshold in thresholds}
        for obs in self.world.generate_all_obs():
            speaker_dist_high = self.speaker.dist_over_utterances_obs(obs, "high")
            speaker_dist_low = self.speaker.dist_over_utterances_obs(obs, "low")
            prob = self.world.obs_prob(obs, self.world.theta)
            for u in self.semantics.utterance_space():
                for threshold in thresholds:
                    if self.get_suspicion(u) >= threshold:
                        tprs[threshold] += speaker_dist_high[u] * prob * pers_ratio
                        tprs[threshold] += speaker_dist_low[u] * prob * (1 - pers_ratio)
        return tprs

def Game(thetas, psis, semantics, world, speaker_type="inf", listener_type="inf", alpha=1.0, rounds=1):
    """
    Play a game of communication between a speaker and a listener.
    """
    literal_speaker = Speaker0(thetas, semantics=semantics, world=world)
    literal_listener = Listener0(thetas, literal_speaker, semantics=semantics, world=world)
    pragmatic_speaker_1 = Speaker1(thetas, literal_listener, semantics=semantics, world=world, alpha=alpha, psi=speaker_type)
    pragmatic_listener_1 = Listener1(thetas, psis, pragmatic_speaker_1, semantics=semantics, world=world, alpha=alpha, listener_type=listener_type)
    pragmatic_speaker_2 = Speaker2(thetas, pragmatic_listener_1, semantics=semantics, world=world, alpha=alpha, psi=speaker_type)
    
    for r in range(rounds):
        print(f"--- Round {r+1} ---")
        obs = world.sample_obs()
        print(f"World theta: {world.theta}, observation: {obs}")
        
        utt = pragmatic_speaker_2.sample_utterance(obs)
        print(f"Speaker uttered: {utt}")
        
        pragmatic_speaker_2.update(obs)
        pragmatic_listener_1.update(utt)
        pragmatic_speaker_1.update(obs)
        literal_listener.update(utt)
        literal_speaker.update(obs)
    return pragmatic_speaker_2, pragmatic_listener_1, pragmatic_speaker_1, literal_listener, literal_speaker

In [ ]:
def plot_speaker_distribution(distribution, title="Speaker Distribution", alpha=1.0):
    """Plot speaker distribution over utterances (q2, predicate) in a fixed order."""
    utterance_order = [
        ("none", "ineffective"),
        ("none", "effective"),
        ("some", "ineffective"),
        ("some", "effective"),
        ("most", "ineffective"),
        ("most", "effective"),
        ("all", "ineffective"),
        ("all", "effective"),
    ]

    utterances = [f"{q2} {pred}" for (q2, pred) in utterance_order]
    probs = [distribution.get((q2, pred), 0.0) for (q2, pred) in utterance_order]

    plt.figure(figsize=(8, 5))
    bars = plt.barh(utterances, probs, color="steelblue")
    plt.xlabel("Probability")
    if alpha > 0:
        plt.title(f"{title} (α={alpha})")
    else:
        plt.title(title)
    plt.xlim(0, 1)

    # Add text labels
    for bar, prob in zip(bars, probs):
        if prob > 0.01:
            plt.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                     f"{prob:.2f}", va="center", fontsize=9)

    plt.gca().invert_yaxis()  # Keep top-down order
    plt.tight_layout()

In [ ]:
from itertools import product
theta = 0.3
thetas = [0.1 * i for i in range(0, 11)]
psis = ["inf", "high", "low"]
world_parameters = {"n": 1, "m": 7}
alpha = 2.0
rounds = 300
speaker_type = "high"
listener_type = "inf"
threshold = 0.9

world = World(theta, world_parameters, get_obs_prob, generate_all_observations)
quantifiers = ["none", "some", "most", "all"]
predicates = ["effective", "ineffective"]
if world_parameters["n"] > 1:
    utterances = list(product(quantifiers, quantifiers, predicates))
else:
    utterances = list(product(quantifiers, predicates))

utterances.append(("all", "none"))
semantics = Semantics(utterances, utterance_is_true)

literal_speaker = Speaker0(thetas, semantics=semantics, world=world)
literal_listener = Listener0(thetas, literal_speaker, semantics=semantics, world=world)
pragmatic_speaker_1 = Speaker1(thetas, literal_listener, semantics=semantics, world=world, alpha=alpha, psi=speaker_type)
if listener_type == "switch":
    pragmatic_listener_1 = Listener1Switch(thetas, psis, pragmatic_speaker_1, semantics=semantics, world=world, alpha=alpha, threshold=threshold)
else:
    pragmatic_listener_1 = Listener1(thetas, psis, pragmatic_speaker_1, semantics=semantics, world=world, alpha=alpha, listener_type=listener_type)
pragmatic_speaker_2 = Speaker2(thetas, pragmatic_listener_1, semantics=semantics, world=world, alpha=alpha, psi=speaker_type)

obs = (0,0,0,0,0,0,0,1)
pretty_print("pragmatic speaker 1 high - 7", pragmatic_speaker_1.dist_over_utterances_obs(obs, "high"))
pretty_print("pragmatic speaker 1 inf - 7", pragmatic_speaker_1.dist_over_utterances_obs(obs, "inf"))


plot_speaker_distribution(pragmatic_speaker_1.dist_over_utterances_obs(obs, "high"), title="Pragmatic Speaker 1 (high)", alpha=alpha)



pretty_print("pragmatic speaker 1 high - 7", pragmatic_speaker_1.get_persuasiveness("high", obs))
pretty_print("pragmatic speaker 1 inf - 7", pragmatic_speaker_1.get_persuasiveness("inf", obs))



inf_probs = pragmatic_speaker_2.dist_over_utterances_obs(obs, "inf")
high_probs = pragmatic_speaker_2.dist_over_utterances_obs(obs, "high")

def find_mean(theta_dist):
    mean = 0
    for theta, prob in theta_dist.items():
        mean += theta * prob
    return mean

inf_mean = 0
high_mean = 0

for utt, prob in inf_probs.items():
    inf_mean += find_mean(pragmatic_listener_1.infer_state(utt).marginal(0)) * prob
    high_mean += find_mean(pragmatic_listener_1.infer_state(utt).marginal(0)) * high_probs[utt]

print("inf mean:", inf_mean)
print("high mean:", high_mean)

# utt = ('some', 'effective')
# inf_listener_probs = pragmatic_listener_1.infer_state(utt).marginal(0)
# pragmatic_listener_1_vig = Listener1(thetas, psis, pragmatic_speaker_1, semantics=semantics, world=world, alpha=alpha, listener_type="vig")
# vig_listener_probs = pragmatic_listener_1_vig.infer_state(utt).marginal(0)

# pretty_print("inf listener probs:", inf_listener_probs)
# pretty_print("vig listener probs:", vig_listener_probs)
# pretty_print("vig listener types:", pragmatic_listener_1_vig.infer_state(utt).marginal(1))


In [ ]:
obs = (0,0,0,0,0,0,0,1)

title = "Pragmatic Speaker 1 vs Literal Listener (Obs: 7 effective)"
speaker_high_probs = pragmatic_speaker_1.dist_over_utterances_obs(obs, "high")
speaker_inf_probs = pragmatic_speaker_1.dist_over_utterances_obs(obs, "inf")

utterance_order = [
    ("none", "ineffective"),
    ("none", "effective"),
    ("some", "ineffective"),
    ("some", "effective"),
    ("most", "ineffective"),
    ("most", "effective"),
    ("all", "ineffective"),
    ("all", "effective"),
    ("all", "none")
]

utterances = [f"{q2} {pred}" for (q2, pred) in utterance_order]
probs_high = [speaker_high_probs.get((q2, pred), 0.0) for (q2, pred) in utterance_order]
probs_inf = [speaker_inf_probs.get((q2, pred), 0.0) for (q2, pred) in utterance_order]

# numeric positions for y axis
y = np.arange(len(utterances))
height = 0.35  # thickness of each bar

plt.figure(figsize=(9, 6))

bars_high = plt.barh(y - height/2, probs_high, height, color="steelblue", label="high")
bars_inf  = plt.barh(y + height/2, probs_inf,  height, color="lightblue", label="inf")

plt.xlabel("Probability")
plt.yticks(y, utterances)
plt.xlim(0, 1)
if alpha > 0:
    plt.title(f"{title} (α={alpha})")
else:
    plt.title(title)

plt.legend()

# Add text labels
for bars, probs in [(bars_high, probs_high), (bars_inf, probs_inf)]:
    for bar, prob in zip(bars, probs):
        if prob > 0.01:
            plt.text(bar.get_width() + 0.01,
                     bar.get_y() + bar.get_height()/2,
                     f"{prob:.2f}",
                     va="center", fontsize=9)

plt.gca().invert_yaxis()  # Keep top-down order
plt.tight_layout()
plt.show()

In [ ]:
pragmatic_listener_1_inf = Listener1(thetas, psis, pragmatic_speaker_1, semantics=semantics, world=world, alpha=alpha, listener_type="inf")
pragmatic_listener_1_inf.infer_state(('all', 'none')).marginal(0)

In [ ]:
exp_lit_speaker = Speaker0(thetas, semantics=semantics, world=world)
exp_lit_listener = Listener0(thetas, exp_lit_speaker, semantics=semantics, world=world)
exp_speaker = Speaker1(thetas, exp_lit_listener, semantics=semantics, world=world, alpha=alpha, psi="inf")

print(exp_speaker.dist_over_utterances_obs((0,0,0,0,0,0,1,0), "inf"))
exp_speaker.update((0,0,0,0,0,0,1,0))
exp_lit_listener.update(("some", "effective"))

exp_speaker.infer_state((0,0,0,0,0,0,0,1)).as_dict()
exp_speaker.listener.infer_state(("all", "effective")).marginal(0)
exp_speaker.dist_over_utterances_obs((0,0,0,0,0,0,0,1), "inf")

In [ ]:
import uuid
import datetime
import json
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

# ----------------------
# Experiment logger
# ----------------------
@dataclass
class ExperimentLogger:
    params: Dict[str, Any]
    id: str = field(default_factory=lambda: str(uuid.uuid4()))
    timestamp: str = field(default_factory=lambda: datetime.datetime.now().isoformat())
    rounds: List[Dict[str, Any]] = field(default_factory=list)
    agent_histories: Dict[str, Optional[List[Any]]] = field(default_factory=dict)  # filled by finalize()

    def log_round(self, round_idx: int, world_theta: Any, observation: Any, utterance: Any, extra: Dict[str, Any] = None):
        """Log the minimal per-round info. Do NOT copy agent.hist here."""
        entry = {
            "round": round_idx,
            "world_theta": world_theta,
            "observation": observation,
            "utterance": utterance
        }
        if extra:
            entry.update(extra)
        self.rounds.append(entry)

    def finalize(self, agents: Dict[str, Any], copy_hist: bool = True):
        """
        Snapshot agents' `hist` attributes once at the end of the experiment.
        This intentionally stores per-agent `hist` only once (not every round).
        """
        for name, agent in agents.items():
            hist = getattr(agent, "hist", None)  # your agents use self.hist = []
            if hist is None:
                self.agent_histories[name] = None
            else:
                self.agent_histories[name] = list(hist) if copy_hist else hist

    # helpers for serializing (safe fallback: str(...) for non-JSON-serializable objects)
    def _make_serializable(self, obj):
        try:
            json.dumps(obj)
            return obj
        except (TypeError, OverflowError):
            return str(obj)

    def to_dict(self, serializable: bool = False) -> Dict[str, Any]:
        meta = {"id": self.id, "timestamp": self.timestamp, "params": self.params}
        rounds = self.rounds
        agent_histories = self.agent_histories
        if serializable:
            meta["params"] = {k: self._make_serializable(v) for k, v in self.params.items()}
            rounds = [{k: self._make_serializable(v) for k, v in r.items()} for r in rounds]
            agent_histories = {k: ([self._make_serializable(v) for v in hist] if hist is not None else None)
                               for k, hist in agent_histories.items()}
        return {"meta": meta, "rounds": rounds, "agent_histories": agent_histories}

    def to_json(self, filename: Optional[str] = None, serializable: bool = True) -> Optional[str]:
        """Write to a file (or return json string if filename is None)."""
        d = self.to_dict(serializable=serializable)
        j = json.dumps(d, indent=2)
        if filename:
            with open(filename, "w") as f:
                f.write(j)
            return None
        return j


# ----------------------
# Runner functions
# ----------------------
def run_game(thetas, psis, semantics, world,
             speaker_type="inf", listener_type="inf", alpha=1.0, psi_priors=None, rounds: int = 1, threshold=0.5,
             verbose: bool = False, seed: int = None) -> Tuple[ExperimentLogger, Dict[str, Any]]:
    """
    Modern runner: returns (logger, agents_dict).
    - logger.rounds contains minimal per-round logs (no agent.hist duplication).
    - logger.finalize(...) is called automatically to snapshot agent.hist once.
    - agents_dict contains references to the live agent objects (so caller can inspect agent.hist)
    """
    # initialize agents (your existing constructors)
    if seed is None:
        seed = random.randint(0, 2**32 - 1)
    
    random.seed(seed)
    literal_speaker = Speaker0(thetas, semantics=semantics, world=world)
    literal_listener = Listener0(thetas, literal_speaker, semantics=semantics, world=world)
    pragmatic_speaker_1 = Speaker1(thetas, literal_listener, semantics=semantics, world=world, alpha=alpha, psi=speaker_type)
    if listener_type == "switch":
        pragmatic_listener_1 = Listener1Switch(thetas, psis, pragmatic_speaker_1, semantics=semantics, world=world, alpha=alpha, threshold=threshold)
    else:
        pragmatic_listener_1 = Listener1(thetas, psis, pragmatic_speaker_1, semantics=semantics, world=world, alpha=alpha, listener_type=listener_type)
    if listener_type == "switch" and psi_priors is not None:
        priors = []
        for theta in thetas:
            for psi in psis:
                priors.append(psi_priors[psi] / len(thetas))
        pragmatic_listener_1.state_belief = Belief(list(product(thetas, psis)), priors)
        pragmatic_listener_1.vig_listener.state_belief = Belief(list(product(thetas, psis)), priors)
    pragmatic_speaker_2 = Speaker2(thetas, pragmatic_listener_1, semantics=semantics, world=world, alpha=alpha, psi=speaker_type)

    print(pragmatic_listener_1.state_belief.marginal(1))
    params = {
        "thetas": thetas,
        "psis": psis,
        "semantics": str(semantics),
        "world": str(world),
        "speaker_type": speaker_type,
        "listener_type": listener_type,
        "alpha": alpha,
        "rounds": rounds,
        "seed": seed
    }
    logger = ExperimentLogger(params=params)

    agents = {
        "pragmatic_speaker_2": pragmatic_speaker_2,
        "pragmatic_listener_1": pragmatic_listener_1,
        "pragmatic_speaker_1": pragmatic_speaker_1,
        "literal_listener": literal_listener,
        "literal_speaker": literal_speaker,
    }

    for r in range(1, rounds + 1):
        if verbose:
            print(f"--- Round {r} ---")
        obs = world.sample_obs()
        if verbose:
            print(f"World theta: {world.theta}, observation: {obs}")

        utt = pragmatic_speaker_2.sample_utterance(obs)
        if verbose:
            print(f"Speaker uttered: {utt}")

        # belief updates (your original order)
        # print(pragmatic_speaker_1.get_informativeness_obs(obs))
        
        pragmatic_speaker_2.update(obs)
        pragmatic_listener_1.update(utt)
        pragmatic_speaker_1.update(obs)
        literal_listener.update(utt)
        literal_speaker.update(obs)

        # minimal per-round log (no agent.hist here)
        logger.log_round(r, world.theta, obs, utt)

    # snapshot agent.hist once
    logger.finalize(agents)
    return logger, agents
def run_game_list1(thetas, psis, semantics, world,
             speaker_type="inf", listener_type="inf", alpha=1.0, psi_priors=None, rounds: int = 1, threshold=0.5,
             verbose: bool = False, seed: int = None) -> Tuple[ExperimentLogger, Dict[str, Any]]:
    """
    Modern runner: returns (logger, agents_dict).
    - logger.rounds contains minimal per-round logs (no agent.hist duplication).
    - logger.finalize(...) is called automatically to snapshot agent.hist once.
    - agents_dict contains references to the live agent objects (so caller can inspect agent.hist)
    """
    # initialize agents (your existing constructors)
    if seed is None:
        seed = random.randint(0, 2**32 - 1)
    
    random.seed(seed)
    literal_speaker = Speaker0(thetas, semantics=semantics, world=world)
    literal_listener = Listener0(thetas, literal_speaker, semantics=semantics, world=world)
    pragmatic_speaker_1 = Speaker1(thetas, literal_listener, semantics=semantics, world=world, alpha=alpha, psi=speaker_type)
    if listener_type == "switch":
        pragmatic_listener_1 = Listener1Switch(thetas, psis, pragmatic_speaker_1, semantics=semantics, world=world, alpha=alpha, threshold=threshold)
    else:
        pragmatic_listener_1 = Listener1(thetas, psis, pragmatic_speaker_1, semantics=semantics, world=world, alpha=alpha, listener_type=listener_type)
    if listener_type == "switch" and psi_priors is not None:
        priors = []
        for theta in thetas:
            for psi in psis:
                priors.append(psi_priors[psi] / len(thetas))
        pragmatic_listener_1.state_belief = Belief(list(product(thetas, psis)), priors)
        pragmatic_listener_1.vig_listener.state_belief = Belief(list(product(thetas, psis)), priors)
    pragmatic_speaker_2 = Speaker2(thetas, pragmatic_listener_1, semantics=semantics, world=world, alpha=alpha, psi=speaker_type)

    print(pragmatic_listener_1.state_belief.marginal(1))
    params = {
        "thetas": thetas,
        "psis": psis,
        "semantics": str(semantics),
        "world": str(world),
        "speaker_type": speaker_type,
        "listener_type": listener_type,
        "alpha": alpha,
        "rounds": rounds,
        "seed": seed
    }
    logger = ExperimentLogger(params=params)

    agents = {
        "pragmatic_listener_1": pragmatic_listener_1,
        "pragmatic_speaker_1": pragmatic_speaker_1,
        "literal_listener": literal_listener,
        "literal_speaker": literal_speaker,
    }

    for r in range(1, rounds + 1):
        if verbose:
            print(f"--- Round {r} ---")
        obs = world.sample_obs()
        if verbose:
            print(f"World theta: {world.theta}, observation: {obs}")

        utt = pragmatic_speaker_1.sample_utterance(obs)
        if verbose:
            print(f"Speaker uttered: {utt}")

        # belief updates (your original order)
        # print(pragmatic_speaker_1.get_informativeness_obs(obs))
        
        pragmatic_listener_1.update(utt)
        pragmatic_speaker_1.update(obs)
        literal_listener.update(utt)
        literal_speaker.update(obs)

        # minimal per-round log (no agent.hist here)
        logger.log_round(r, world.theta, obs, utt)

    # snapshot agent.hist once
    logger.finalize(agents)
    return logger, agents
def run_game_list2(thetas, psis, semantics, world,
             speaker_type="inf", listener_type="inf", alpha=1.0, psi_priors=None, rounds: int = 1, threshold=0.5,
             verbose: bool = False, seed: int = None) -> Tuple[ExperimentLogger, Dict[str, Any]]:
    """
    Modern runner: returns (logger, agents_dict).
    - logger.rounds contains minimal per-round logs (no agent.hist duplication).
    - logger.finalize(...) is called automatically to snapshot agent.hist once.
    - agents_dict contains references to the live agent objects (so caller can inspect agent.hist)
    """
    # initialize agents (your existing constructors)
    if seed is None:
        seed = random.randint(0, 2**32 - 1)
    
    random.seed(seed)
    literal_speaker = Speaker0(thetas, semantics=semantics, world=world)
    literal_listener = Listener0(thetas, literal_speaker, semantics=semantics, world=world)
    pragmatic_speaker_1 = Speaker1(thetas, literal_listener, semantics=semantics, world=world, alpha=alpha, psi=speaker_type)
    if listener_type == "switch":
        pragmatic_listener_1 = Listener1Switch(thetas, psis, pragmatic_speaker_1, semantics=semantics, world=world, alpha=alpha, threshold=threshold)
    else:
        pragmatic_listener_1 = Listener1(thetas, psis, pragmatic_speaker_1, semantics=semantics, world=world, alpha=alpha, listener_type=listener_type)
    if listener_type == "switch" and psi_priors is not None:
        priors = []
        for theta in thetas:
            for psi in psis:
                priors.append(psi_priors[psi] / len(thetas))
        pragmatic_listener_1.state_belief = Belief(list(product(thetas, psis)), priors)
        pragmatic_listener_1.vig_listener.state_belief = Belief(list(product(thetas, psis)), priors)
    pragmatic_speaker_2 = Speaker2(thetas, pragmatic_listener_1, semantics=semantics, world=world, alpha=alpha, psi=speaker_type)
    
    if listener_type == "switch":
        pragmatic_listener_2 = Listener2Switch(thetas, psis, pragmatic_speaker_2, semantics=semantics, world=world, alpha=alpha, threshold=threshold)
    elif listener_type == "inf":
        pragmatic_listener_2 = Listener1(thetas, psis, pragmatic_speaker_2, semantics=semantics, world=world, alpha=alpha, listener_type="inf")
    elif listener_type == "vig":
        pragmatic_listener_2 = Listener1(thetas, psis, pragmatic_speaker_2, semantics=semantics, world=world, alpha=alpha, listener_type="vig")
    
    print(pragmatic_listener_1.state_belief.marginal(1))
    params = {
        "thetas": thetas,
        "psis": psis,
        "semantics": str(semantics),
        "world": str(world),
        "speaker_type": speaker_type,
        "listener_type": listener_type,
        "alpha": alpha,
        "rounds": rounds,
        "seed": seed
    }
    logger = ExperimentLogger(params=params)

    agents = {
        "pragmatic_listener_2": pragmatic_listener_2,
        "pragmatic_speaker_2": pragmatic_speaker_2,
        "pragmatic_listener_1": pragmatic_listener_1,
        "pragmatic_speaker_1": pragmatic_speaker_1,
        "literal_listener": literal_listener,
        "literal_speaker": literal_speaker,
    }

    for r in range(1, rounds + 1):
        if verbose:
            print(f"--- Round {r} ---")
        obs = world.sample_obs()
        if verbose:
            print(f"World theta: {world.theta}, observation: {obs}")

        utt = pragmatic_speaker_2.sample_utterance(obs)
        if verbose:
            print(f"Speaker uttered: {utt}")

        # belief updates (your original order)
        # print(pragmatic_speaker_1.get_informativeness_obs(obs))
        pragmatic_listener_2.update(utt)
        pragmatic_speaker_2.update(obs)
        pragmatic_listener_1.update(utt)
        pragmatic_speaker_1.update(obs)
        literal_listener.update(utt)
        literal_speaker.update(obs)

        # minimal per-round log (no agent.hist here)
        logger.log_round(r, world.theta, obs, utt)

    # snapshot agent.hist once
    logger.finalize(agents)
    return logger, agents

# ----------------------
# Backwards-compatible wrapper
# ----------------------
def Game(thetas, psis, semantics, world,
         speaker_type="inf", listener_type="inf", alpha=1.0, rounds: int = 1, verbose: bool = False):
    """
    Backwards-compatible Game(...) which returns the same five agents as before,
    BUT also attaches the ExperimentLogger to the top-level returned speaker as
    `pragmatic_speaker_2.experiment_log` for convenience.
    Old call-pattern:
        ps2, pl1, ps1, ll, ls = Game(...)
    New convenience:
        ps2.experiment_log  # contains ExperimentLogger with rounds + final agent hist snapshots
    """
    logger, agents = run_game(thetas, psis, semantics, world,
                              speaker_type=speaker_type, listener_type=listener_type,
                              alpha=alpha, rounds=rounds, verbose=verbose)
    # attach logger to the returned top-level speaker for backward compat
    agents["pragmatic_speaker_2"].experiment_log = logger
    # return agents in the old tuple order (so your existing code doesn't break)
    return (agents["pragmatic_listener_2"],
            agents["pragmatic_speaker_2"],
            agents["pragmatic_listener_1"],
            agents["pragmatic_speaker_1"],
            agents["literal_listener"],
            agents["literal_speaker"])

In [ ]:
from scipy.special import binom
import math
from functools import lru_cache

def generate_all_observations(world_parameters):
    """
    Generate all possible observation histograms for n patients and m sessions.
    Each histogram is a tuple of length m+1, summing to n.
    """
    n = world_parameters["n"]
    m = world_parameters["m"]
    observations = []

    def helper(current, depth, remaining):
        if depth == m:
            current.append(remaining)
            observations.append(tuple(current))
            current.pop()
            return
        for i in range(remaining + 1):
            current.append(i)
            helper(current, depth + 1, remaining - i)
            current.pop()

    helper([], 0, n)
    return observations

@lru_cache(maxsize=None)
def utterance_is_true(u, obs):
    """Evaluate if utterance u = (q1, q2, pred) is true given obs[n x m]"""
    if u == ("all", "none"):
        return obs[0] == sum(obs) or obs[-1] == sum(obs)
    
    if sum(obs) > 1:
        q1, q2, pred = u
        n = sum(obs)
        m = len(obs) - 1
        k = 0
        if pred == "ineffective":
            obs = obs[::-1]  # Reverse obs to treat "ineffective" as the last element
        
        # Step 1: apply q2 to each patient
        if q2 == "none":
            k = obs[0]
        elif q2 == "some":
            k = sum(obs[1:])
        elif q2 == "most":
            k = sum(obs[math.floor(m / 2) + (m % 2) : ])
        elif q2 == "all":
            k = obs[-1]

        # Step 2: apply q1 across patients
        if q1 == "none":
            return k == 0
        elif q1 == "some":
            return k >= 1
        elif q1 == "most":
            return k > (n / 2)
        elif q1 == "all":
            return k == n
    else:
        """Evaluate if utterance u = (q2, pred) is true given obs[1 x m]"""
        q2, pred = u
        m = len(obs) - 1
        patient_score = obs.index(1) if pred == "effective" else m - obs.index(1)

        if q2 == "none":
            return patient_score == 0
        elif q2 == "some":
            return patient_score > 0
        elif q2 == "most":
            return patient_score > (m / 2)
        elif q2 == "all":
            return patient_score == m

@lru_cache(maxsize=None)  
def multinomial(params):
    if len(params) == 1:
        return 1
    return binom(sum(params), params[-1]) * multinomial(params[:-1])

@lru_cache(maxsize=None)
def get_obs_prob(obs, theta):
    #print("lan", obs)
    n = sum(obs)
    m = len(obs) - 1
    flat_prob = 1
    def helper(effective):
        return math.comb(m, effective) * (theta ** effective) * ((1 - theta) ** (m - effective))
    for i in range(len(obs)):
        flat_prob *= helper(i) ** obs[i]
    return flat_prob * multinomial(obs)

In [ ]:
from itertools import product
theta = 0.4
thetas = [0.1 * i for i in range(0, 11)]
psis = ["inf", "high", "low"]
world_parameters = {"n": 1, "m": 7}
alpha = 3.0
rounds = 500
speaker_type = "high"
listener_type = "switch"
threshold = 0.9

world = World(theta, world_parameters, get_obs_prob, generate_all_observations)
quantifiers = ["none", "some", "most", "all"]
predicates = ["effective", "ineffective"]
if world_parameters["n"] > 1:
    utterances = list(product(quantifiers, quantifiers, predicates))
else:
    utterances = list(product(quantifiers, predicates))

semantics = Semantics(utterances, utterance_is_true)

logger, agents = run_game(thetas = thetas, psis = psis, semantics = semantics, world = world, alpha=alpha, rounds=rounds, speaker_type=speaker_type, listener_type=listener_type, threshold=threshold, verbose=False)

print(agents["pragmatic_listener_1"].vig_listener.state_belief.marginal(0))
print(agents["pragmatic_listener_1"].inf_listener.state_belief.marginal(0))
print(agents["pragmatic_listener_1"].state_belief.marginal(0))
print(agents["pragmatic_listener_1"].switched)
print(agents["pragmatic_listener_1"].suspicion)
print(agents["pragmatic_listener_1"].switch_point)
# for r in logger.rounds:
#     print(r["utterance"])
# for hist in agents["pragmatic_speaker_1"].hist:
#     print(f"Low: {hist.as_dict()[0.1]}, High: {hist.as_dict()[0.8]}")

print("\n")


In [ ]:
from itertools import product
import matplotlib.pyplot as plt
theta = 0.3
thetas = [0.1 * i for i in range(0, 11)]
psis = ["inf", "high"]
world_parameters = {"n": 1, "m": 7}
alpha = 3
rounds = 500
speaker_type = "high"
listener_type = "switch"
threshold = 0.9

world = World(theta, world_parameters, get_obs_prob, generate_all_observations)
quantifiers = ["none", "some", "most", "all"]
predicates = ["effective", "ineffective"]
if world_parameters["n"] > 1:
    utterances = list(product(quantifiers, quantifiers, predicates))
else:
    utterances = list(product(quantifiers, predicates))

semantics = Semantics(utterances, utterance_is_true)

literal_speaker = Speaker0(thetas, semantics=semantics, world=world)
literal_listener = Listener0(thetas, literal_speaker, semantics=semantics, world=world)
pragmatic_speaker_1 = Speaker1(thetas, literal_listener, semantics=semantics, world=world, alpha=alpha, psi="high")
pragmatic_listener_1 = Listener1(thetas, psis, pragmatic_speaker_1, semantics=semantics, world=world, alpha=alpha, listener_type="inf", threshold=threshold)
pragmatic_listener_1_vig = Listener1(thetas, psis, pragmatic_speaker_1, semantics=semantics, world=world, alpha=alpha, listener_type="vig", threshold=threshold)

pragmatic_speaker_2_inf = Speaker2(thetas, pragmatic_listener_1, semantics=semantics, world=world, alpha=alpha, psi="high")
pragmatic_speaker_2_vig = Speaker2(thetas, pragmatic_listener_1_vig, semantics=semantics, world=world, alpha=alpha, psi="high")

utt = ('most', 'effective')
obs = (0,0,0,0,1,0,0,0)

print(pragmatic_listener_1.infer_obs(utt))

probs_inf = [prob for obs, prob in pragmatic_listener_1.infer_obs(utt).items()]
probs_vig = [prob for obs, prob in pragmatic_listener_1_vig.infer_obs(utt).items()]
print(pragmatic_listener_1_vig.psis)

# print(pragmatic_speaker_2_inf.get_persuasiveness("high"), obs)
# print(pragmatic_speaker_2_vig.get_persuasiveness("high"), obs)

print(pragmatic_speaker_2_inf.dist_over_utterances_obs(obs, "high")[utt])
print(pragmatic_speaker_2_vig.dist_over_utterances_obs(obs, "high")[utt])
print(pragmatic_speaker_2_inf.dist_over_utterances_obs(obs, "high"))
print(pragmatic_speaker_2_vig.dist_over_utterances_obs(obs, "high"))


plt.bar(range(len(probs_inf)), list(reversed(probs_inf)))
fig, ax = plt.subplots()
ax.bar(range(len(probs_vig)), list(reversed(probs_vig)))

In [ ]:
from itertools import product
theta = 0.7
thetas = [0.1 * i for i in range(0, 11)]
psis = ["inf", "high", "low"]
world_parameters = {"n": 1, "m": 7}
alpha = 3.0
rounds = 600
speaker_type = "high"
listener_type = "switch"
threshold = 0.4

world = World(theta, world_parameters, get_obs_prob, generate_all_observations)
quantifiers = ["none", "some", "most", "all"]
predicates = ["effective", "ineffective"]
if world_parameters["n"] > 1:
    utterances = list(product(quantifiers, quantifiers, predicates))
else:
    utterances = list(product(quantifiers, predicates))

semantics = Semantics(utterances, utterance_is_true)

logger, agents = run_game_list2(thetas = thetas, psis = psis, semantics = semantics, world = world, alpha=alpha, rounds=rounds, speaker_type=speaker_type, listener_type=listener_type, threshold=threshold, verbose=False, seed = None, psi_priors = {"inf": 1, "high": 1, "low": 1})
logger_vig, agents_vig = run_game_list2(thetas = thetas, psis = psis, semantics = semantics, world = world, alpha=alpha, rounds=rounds, speaker_type=speaker_type, listener_type="vig", threshold=threshold, verbose=False, seed = logger.params["seed"])
logger_inf, agents_inf = run_game_list2(thetas = thetas, psis = psis, semantics = semantics, world = world, alpha=alpha, rounds=rounds, speaker_type=speaker_type, listener_type="inf", threshold=threshold, verbose=False, seed = logger.params["seed"])

In [ ]:
print(f"Obs: {logger.rounds[0]['observation']}, Utt: {logger.rounds[0]['utterance']}")

In [ ]:
import pprint

def pretty_print(title, obj, precision=4):
    print(f"\n=== {title} ===")
    
    if isinstance(obj, dict):
        for k, v in obj.items():
            if isinstance(v, (float, np.floating)):
                print(f"{str(k):>20}: {float(v):.{precision}g}")
            else:
                print(f"{str(k):>20}: {v}")
    elif isinstance(obj, (list, tuple)):
        for i, v in enumerate(obj):
            if isinstance(v, (float, np.floating)):
                print(f"[{i}] {float(v):.{precision}g}")
            else:
                print(f"[{i}] {v}")
    else:
        pprint.pprint(obj)

In [ ]:
agents_vig["pragmatic_listener_1"].hist[0].marginal(1)

In [ ]:
obs = (0,0,1,0,0,0,0,0)

# pretty_print("Persuasiveness (inf)", agents_inf["pragmatic_speaker_2"].get_persuasiveness("high", obs))
# pretty_print("Persuasiveness (vig)", agents_vig["pragmatic_speaker_2"].get_persuasiveness("high", obs))

# pretty_print("Distribution (inf)", agents_inf["pragmatic_speaker_2"].dist_over_utterances_obs(obs, "high"))
# pretty_print("Distribution (vig)", agents_vig["pragmatic_speaker_2"].dist_over_utterances_obs(obs, "high"))

# pretty_print("Listener belief (state=1)", agents_vig["pragmatic_listener_1"].state_belief.marginal(1))
# pretty_print("Listener belief (state=0)", agents_vig["pragmatic_listener_1"].state_belief.marginal(0))

agents_inf["pragmatic_speaker_2"].get_persuasiveness("high", obs, debugger=True)

In [ ]:
print(agents["pragmatic_listener_1"].switch_point)

utt = ("most", "effective")

print(agents_inf["pragmatic_speaker_2"].prob_hist[-1])
a,b,c = [], [], []

for i in range(len(agents["pragmatic_speaker_2"].prob_hist)):
    a.append(agents["pragmatic_speaker_2"].prob_hist[i][utt])
    b.append(agents_vig["pragmatic_speaker_2"].prob_hist[i][utt])
    c.append(agents_inf["pragmatic_speaker_2"].prob_hist[i][utt])
            

plt.plot(a, label="switch")
plt.plot(b, label="vig")
plt.plot(c, label="inf")
plt.legend()
plt.xlabel("Rounds")

In [ ]:
utt = ("some", "effective")


a,b,c = [], [], []

for i in range(len(agents["pragmatic_speaker_2"].prob_hist)):
    a.append(agents["pragmatic_speaker_2"].prob_hist[i][utt])
    b.append(agents_vig["pragmatic_speaker_2"].prob_hist[i][utt])
    c.append(agents_inf["pragmatic_speaker_2"].prob_hist[i][utt])
            

plt.plot(a, label="switch")
plt.plot(b, label="vig")
plt.plot(c, label="inf")
plt.legend()
plt.xlabel("Rounds")

In [ ]:
i = 0
for round in logger.rounds[450:500]:
    print(f"round {round['round']}: {round['utterance']}, obs: {round['observation'].index(1)},  belief: {agents['pragmatic_listener_1'].vig_listener.hist[i].marginal(1)}, theta: {find_mean(agents['pragmatic_listener_1'].inf_listener.hist[i].marginal(0))} , sus: {agents['pragmatic_listener_1'].suspicion[i]}")
    i+=1

In [ ]:
i = 0
for round in logger_vig.rounds[0:60]:
    print(f"round {round['round']}: {round['utterance']}, obs: {round['observation'].index(1)},  belief: {agents_vig['pragmatic_listener_1'].hist[i].marginal(1)} , sus: {agents_vig['pragmatic_listener_1'].suspicion[i]}")
    i+=1

In [ ]:
i = 0
for round in logger_inf.rounds[0:150]:
    print(f"round {round['round']}: {round['utterance']}, obs: {round['observation'].index(1)},  belief: {find_mean(agents_inf['pragmatic_listener_1'].hist[i].marginal(0))} , sus: {agents_inf['pragmatic_listener_1'].suspicion[i]}")
    i+=1

In [ ]:
i = 0
for round in logger_inf.rounds[0:150]:
    print(f"round {round['round']}: {round['utterance']}, obs: {round['observation'].index(1)},  belief: {find_mean(agents['literal_listener'].hist[i].marginal(0))} , switch_belief: {find_mean(agents['pragmatic_listener_1'].hist[i].marginal(0))} , sus: {agents_inf['pragmatic_listener_1'].suspicion[i]}")
    i+=1

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def summarize_beliefs(belief_hist):
    thetas = np.array(sorted(belief_hist[0].keys()))
    means, ses = [], []
    for dist in belief_hist:
        ps = np.array([dist[th] for th in thetas])
        mean = np.sum(thetas * ps)
        var = np.sum((thetas - mean)**2 * ps)
        std = np.sqrt(var)
        # SE = std / sqrt(N); here N=1 distribution per round, so just use std as uncertainty
        se = std
        means.append(mean)
        ses.append(se)
    return np.arange(1, len(belief_hist)+1), np.array(means), np.array(ses)

switch_theta_beliefs = []
vig_theta_beliefs = []
inf_theta_beliefs = []
for belief in agents["pragmatic_listener_2"].hist:
    switch_theta_beliefs.append(belief.marginal(0))

for belief in agents["pragmatic_listener_2"].vig_listener.hist:
    vig_theta_beliefs.append(belief.marginal(0))

for belief in agents["pragmatic_listener_2"].inf_listener.hist:
    inf_theta_beliefs.append(belief.marginal(0))

switch_rounds, switch_means, switch_ses = summarize_beliefs(switch_theta_beliefs)
vig_rounds, vig_means, vig_ses = summarize_beliefs(vig_theta_beliefs)
inf_rounds, inf_means, inf_ses = summarize_beliefs(inf_theta_beliefs)

plt.figure(figsize=(8,5))
plt.plot(switch_rounds, switch_means, label="mean θ (Switch)", color="blue")
plt.fill_between(switch_rounds, switch_means - switch_ses, switch_means + switch_ses, color="blue", alpha=0.2, label="± std")

plt.plot(vig_rounds, vig_means, label="mean θ (Vig)", color="orange")
plt.fill_between(vig_rounds, vig_means - vig_ses, vig_means + vig_ses, color="orange", alpha=0.2, label="± std")

plt.plot(inf_rounds, inf_means, label="mean θ (Nai)", color="green")
plt.fill_between(inf_rounds, inf_means - inf_ses, inf_means + inf_ses, color="green", alpha=0.2, label="± std")

plt.axvline(x=agents["pragmatic_listener_2"].switch_point, color='r', linestyle='--', linewidth=2)
plt.ylim(0, 1)
plt.xlabel("Round")
plt.ylabel("θ belief")
plt.title(r"$L_1$ vs $S_2$: $\psi = $ high | $\theta = 0.3$ | $\alpha = 3$ ")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
# plt.text(1.12, 0.5, r"$\alpha = \frac{1}{1+\beta}$", transform=plt.gca().transAxes,
#          fontsize=12, va="center")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
print(agents["pragmatic_listener_2"].switch_point)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def summarize_beliefs(belief_hist):
    thetas = np.array(sorted(belief_hist[0].keys()))
    means, ses = [], []
    for dist in belief_hist:
        ps = np.array([dist[th] for th in thetas])
        mean = np.sum(thetas * ps)
        var = np.sum((thetas - mean)**2 * ps)
        std = np.sqrt(var)
        # SE = std / sqrt(N); here N=1 distribution per round, so just use std as uncertainty
        se = std
        means.append(mean)
        ses.append(se)
    return np.arange(1, len(belief_hist)+1), np.array(means), np.array(ses)

switch_theta_beliefs = []
vig_theta_beliefs = []
inf_theta_beliefs = []
for belief in agents["pragmatic_listener_2"].hist:
    switch_theta_beliefs.append(belief.marginal(0))

for belief in agents_vig["pragmatic_listener_2"].hist:
    vig_theta_beliefs.append(belief.marginal(0))

for belief in agents_inf["pragmatic_listener_2"].hist:
    inf_theta_beliefs.append(belief.marginal(0))

switch_rounds, switch_means, switch_ses = summarize_beliefs(switch_theta_beliefs)
vig_rounds, vig_means, vig_ses = summarize_beliefs(vig_theta_beliefs)
inf_rounds, inf_means, inf_ses = summarize_beliefs(inf_theta_beliefs)

plt.figure(figsize=(8,5))
plt.plot(switch_rounds, switch_means, label="mean θ (Switch)", color="blue")
plt.fill_between(switch_rounds, switch_means - switch_ses, switch_means + switch_ses, color="blue", alpha=0.2, label="± std")

plt.plot(vig_rounds, vig_means, label="mean θ (Vig)", color="orange")
plt.fill_between(vig_rounds, vig_means - vig_ses, vig_means + vig_ses, color="orange", alpha=0.2, label="± std")

plt.plot(inf_rounds, inf_means, label="mean θ (Nai)", color="green")
plt.fill_between(inf_rounds, inf_means - inf_ses, inf_means + inf_ses, color="green", alpha=0.2, label="± std")

plt.axvline(x=agents["pragmatic_listener_2"].switch_point, color='r', linestyle='--', linewidth=2)
plt.ylim(0, 1)
plt.xlabel("Round")
plt.ylabel("θ belief")
plt.title(r"$L_1$ vs $S_2$: $\psi = $ high | $\theta = 0.3$ | $\alpha = 3$ ")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
# plt.text(1.12, 0.5, r"$\alpha = \frac{1}{1+\beta}$", transform=plt.gca().transAxes,
#          fontsize=12, va="center")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
print(agents["pragmatic_listener_2"].switch_point)
print(agents["pragmatic_listener_2"].vig_listener.switch_point)
print(agents["pragmatic_listener_1"].switch_point)

In [ ]:
def find_pred(logger, pred):
    total = 0
    for round in logger.rounds:
        total += round["utterance"].count(pred)
    return total

def find_pred_cum(logger, pred):
    total_cum = []
    total = 0
    for round in logger.rounds:
        total += round["utterance"].count(pred)
        total_cum.append(total)
    return total_cum

def find_utt(logger, utt):
    total = 0
    for round in logger.rounds:
        total += round["utterance"] == utt
    return total

def find_utt_cum(logger, utt):
    total_cum = []
    total = 0
    for round in logger.rounds:
        total += round["utterance"] == utt
        total_cum.append(total)
    return total_cum

u_check = ("some", "effective")
u_check_2 = ("some", "ineffective")
plt.plot(find_utt_cum(logger, u_check), label=f"switch: {u_check}")
plt.plot(find_utt_cum(logger_vig, u_check), label=f"vig: {u_check}")

plt.plot(find_utt_cum(logger, u_check_2), label=f"switch: {u_check_2}")
plt.plot(find_utt_cum(logger_vig, u_check_2), label=f"vig: {u_check_2}")

print(f"Switch {u_check}:", find_utt(logger, u_check))

print(f"Vig {u_check}:", find_utt(logger_vig, u_check))

print(f"Inf {u_check}:", find_utt(logger_inf, u_check))


# pred = "some"
# plt.plot(find_pred_cum(logger_vig, pred), label=f"vig: {pred}")
# plt.plot(find_pred_cum(logger, pred), label=f"switch: {pred}")

# pred = "most"
# plt.plot(find_pred_cum(logger_vig, pred), label=f"vig: {pred}")
# plt.plot(find_pred_cum(logger, pred), label=f"switch: {pred}")

plt.legend()
print(f"Switch {pred}:", find_pred(logger, pred))

print(f"Vig {pred}:", find_pred(logger_vig, pred))

print(f"Inf {pred}:", find_pred(logger_inf, pred))

In [ ]:
agents_vig["pragmatic_listener_1"].state_belief.marginal(1)
agents["pragmatic_listener_1"].switch_point

In [ ]:
def plot_multiple_dists(dists, labels, title="Distributions over utterances"):
    """
    dists: list of dicts {utterance: prob}
    labels: list of labels (e.g. psi values) for each distribution
    """
    utterances = list(dists[0].keys())  # assume same utterances across dists
    x = np.arange(len(utterances))
    width = 0.8 / len(dists)  # bar width depends on number of dists
    
    plt.figure(figsize=(10, 6))
    
    for i, dist in enumerate(dists):
        probs = [dist[utt] for utt in utterances]
        plt.bar(x + i * width, probs, width, label=labels[i])
    
    plt.ylim(0, 1)
    plt.xticks(x + width * (len(dists) - 1) / 2, utterances, rotation=45, ha="right")
    plt.xlabel("Utterances")
    plt.ylabel("Probability")
    plt.title(title)
    plt.legend(title="Distributions")
    plt.grid(axis="y", linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.show()


In [ ]:
threshold = 0.6

literal_speaker = Speaker0(thetas, semantics=semantics, world=world)
literal_listener = Listener0(thetas, literal_speaker, semantics=semantics, world=world)
pragmatic_speaker_1 = Speaker1(thetas, literal_listener, semantics=semantics, world=world, alpha=alpha, psi=speaker_type)
pragmatic_listener_1_inf = Listener1(thetas, psis, pragmatic_speaker_1, semantics=semantics, world=world, alpha=alpha, listener_type="inf")
pragmatic_listener_1_vig = Listener1(thetas, psis, pragmatic_speaker_1, semantics=semantics, world=world, alpha=alpha, listener_type="vig")
pragmatic_listener_1_switch = Listener1Switch(thetas, psis, pragmatic_speaker_1, semantics=semantics, world=world, alpha=alpha, threshold=threshold)

pragmatic_speaker_2_inf = Speaker2(thetas, pragmatic_listener_1_inf, semantics=semantics, world=world, alpha=alpha, psi=speaker_type)
pragmatic_speaker_2_vig = Speaker2(thetas, pragmatic_listener_1_vig, semantics=semantics, world=world, alpha=alpha, psi=speaker_type)
pragmatic_speaker_2_switch = Speaker2(thetas, pragmatic_listener_1_switch, semantics=semantics, world=world, alpha=alpha, psi=speaker_type)

vig_result = pragmatic_listener_1_vig.infer_state(("some", "effective")).marginal(0)
inf_result = pragmatic_listener_1_inf.infer_state(("some", "effective")).marginal(0)
switch_result = pragmatic_listener_1_switch.infer_state(("some", "effective")).marginal(0)

plt.plot(thetas, vig_result.values(), marker="o")
#plt.plot(thetas, inf_result.values(), marker="o")
plt.plot(thetas, switch_result.values(), marker="o")
utt = ("most", "effective")
vig_result = pragmatic_listener_1_vig.infer_state(utt).marginal(0)
inf_result = pragmatic_listener_1_inf.infer_state(utt).marginal(0)
switch_result = pragmatic_listener_1_switch.infer_state(utt).marginal(0)
#new plot
fig1, ax1 = plt.subplots()
ax1.plot(thetas, vig_result.values(), marker="o", label="vig")
#ax1.plot(thetas, inf_result.values(), marker="o", label="inf")
ax1.plot(thetas, switch_result.values(), marker="o", label="switch")

for obs in world.generate_all_obs():
    dists = []
    dists.append(pragmatic_speaker_2_inf.dist_over_utterances_obs(obs, speaker_type))
    dists.append(pragmatic_speaker_2_vig.dist_over_utterances_obs(obs, speaker_type))
    dists.append(pragmatic_speaker_2_switch.dist_over_utterances_obs(obs, speaker_type))
    labels = ["inf", "vig", "switch"]
    plot_multiple_dists(dists, labels, title=f"Utterance distributions for obs {obs}")


In [ ]:
threshold = 0.9

literal_speaker = Speaker0(thetas, semantics=semantics, world=world)
literal_listener = Listener0(thetas, literal_speaker, semantics=semantics, world=world)
pragmatic_speaker_1 = Speaker1(thetas, literal_listener, semantics=semantics, world=world, alpha=alpha, psi=speaker_type)
pragmatic_listener_1_inf = Listener1(thetas, psis, pragmatic_speaker_1, semantics=semantics, world=world, alpha=alpha, listener_type="inf")
pragmatic_listener_1_vig = Listener1(thetas, psis, pragmatic_speaker_1, semantics=semantics, world=world, alpha=alpha, listener_type="vig")
pragmatic_listener_1_switch = Listener1(thetas, psis, pragmatic_speaker_1, semantics=semantics, world=world, alpha=alpha, listener_type="switch", threshold=threshold)

pragmatic_speaker_2_inf = Speaker2(thetas, pragmatic_listener_1_inf, semantics=semantics, world=world, alpha=alpha, psi=speaker_type)
pragmatic_speaker_2_vig = Speaker2(thetas, pragmatic_listener_1_vig, semantics=semantics, world=world, alpha=alpha, psi=speaker_type)
pragmatic_speaker_2_switch = Speaker2(thetas, pragmatic_listener_1_switch, semantics=semantics, world=world, alpha=alpha, psi=speaker_type)


for obs in world.generate_all_obs():
    dists = []
    dists.append(pragmatic_speaker_2_inf.dist_over_utterances_obs(obs, speaker_type))
    dists.append(pragmatic_speaker_2_vig.dist_over_utterances_obs(obs, speaker_type))
    dists.append(pragmatic_speaker_2_switch.dist_over_utterances_obs(obs, speaker_type))
    labels = ["inf", "vig", "switch"]
    plot_multiple_dists(dists, labels, title=f"Utterance distributions for obs {obs}")

# print(pragmatic_listener_1.get_suspicion(("some", "effective")))
# print(pragmatic_listener_1.false_positive_rate(threshold = threshold))
# print(pragmatic_listener_1.true_positive_rate(threshold = threshold, pers_ratio = 0.5))

In [ ]:
theta = 0.5
thetas = [0.1 * i for i in range(0, 11)]
psis = ["inf", "high", "low"]
world_parameters = {"n": 5, "m": 7}
alpha = 30.0
rounds = 1
speaker_type = "low"
listener_type = "vig"

world = World(theta, world_parameters, get_obs_prob, generate_all_observations)
quantifiers = ["none", "some", "most", "all"]
predicates = ["effective", "ineffective"]
if world_parameters["n"] > 1:
    utterances = list(product(quantifiers, quantifiers, predicates))
else:
    utterances = list(product(quantifiers, predicates))

semantics = Semantics(utterances, utterance_is_true)
literal_speaker = Speaker0(thetas, semantics=semantics, world=world)
literal_listener = Listener0(thetas, literal_speaker, semantics=semantics, world=world)

thresholds = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
alphas = [0.5, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 15.0, 20.0, 30.0, 40.0]
#thresholds = [0.2, 0.4, 0.6, 0.8]
#alphas = [1.0]
tprs_1 = {}
fprs_1 = {}
for alpha in alphas:
    pragmatic_speaker_1 = Speaker1(thetas, literal_listener, semantics=semantics, world=world, alpha=alpha, psi=speaker_type)
    pragmatic_listener_1 = Listener1(thetas, psis, pragmatic_speaker_1, semantics=semantics, world=world, alpha=alpha, listener_type=listener_type)
    tprs_t = pragmatic_listener_1.true_positive_rates(thresholds, pers_ratio = 0.5)
    fprs_t = pragmatic_listener_1.false_positive_rates(thresholds)
    for threshold in thresholds:
        tprs_1[(threshold, alpha)] = tprs_t[threshold]
        fprs_1[(threshold, alpha)] = fprs_t[threshold]

In [ ]:
theta = 0.2
thetas = [0.1 * i for i in range(0, 11)]
psis = ["inf", "high", "low"]
world_parameters = {"n": 1, "m": 7}
alpha = 10.0
rounds = 1
speaker_type = "high"
listener_type = "vig"

world = World(theta, world_parameters, get_obs_prob, generate_all_observations)
quantifiers = ["none", "some", "most", "all"]
predicates = ["effective", "ineffective"]
if world_parameters["n"] > 1:
    utterances = list(product(quantifiers, quantifiers, predicates))
else:
    utterances = list(product(quantifiers, predicates))

semantics = Semantics(utterances, utterance_is_true)
literal_speaker = Speaker0(thetas, semantics=semantics, world=world)
literal_listener = Listener0(thetas, literal_speaker, semantics=semantics, world=world)

thresholds = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
alphas = [0.5, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 15.0, 20.0, 30.0, 40.0]
#thresholds = [0.2, 0.4, 0.6, 0.8]
#alphas = [1.0]
tprs = {}
fprs = {}
pers_ratio = 0.5
for alpha in alphas:
    pragmatic_speaker_1 = Speaker1(thetas, literal_listener, semantics=semantics, world=world, alpha=alpha, psi=speaker_type)
    pragmatic_listener_1 = Listener1Switch(thetas, psis, pragmatic_speaker_1, semantics=semantics, world=world, alpha=alpha, threshold=0.5)
    tprs_t = pragmatic_listener_1.true_positive_rates(thresholds, pers_ratio = pers_ratio)
    fprs_t = pragmatic_listener_1.false_positive_rates(thresholds)
    for threshold in thresholds:
        tprs[(threshold, alpha)] = tprs_t[threshold]
        fprs[(threshold, alpha)] = fprs_t[threshold]

In [ ]:
theta = 0.6
thetas = [0.1 * i for i in range(0, 11)]
psis = ["inf", "high", "low"]
world_parameters = {"n": 1, "m": 7}
alpha_low = 3.0
alpha_high = 15.0
rounds = 1
speaker_type = "high"
listener_type = "vig"

world = World(theta, world_parameters, get_obs_prob, generate_all_observations)
quantifiers = ["none", "some", "most", "all"]
predicates = ["effective", "ineffective"]
if world_parameters["n"] > 1:
    utterances = list(product(quantifiers, quantifiers, predicates))
else:
    utterances = list(product(quantifiers, predicates))

semantics = Semantics(utterances, utterance_is_true)
literal_speaker = Speaker0(thetas, semantics=semantics, world=world)
literal_listener = Listener0(thetas, literal_speaker, semantics=semantics, world=world)
pragmatic_speaker_1_low = Speaker1(thetas, literal_listener, semantics=semantics, world=world, alpha=alpha_low, psi=speaker_type)
pragmatic_listener_1_low = Listener1(thetas, psis, pragmatic_speaker_1_low, semantics=semantics, world=world, alpha=alpha_low, threshold=0.5, listener_type = "vig")

pragmatic_speaker_1_high = Speaker1(thetas, literal_listener, semantics=semantics, world=world, alpha=alpha_high, psi=speaker_type)
pragmatic_listener_1_high = Listener1(thetas, psis, pragmatic_speaker_1_high, semantics=semantics, world=world, alpha=alpha_high, threshold=0.5, listener_type = "vig")

suspicions_low = {}
suspicions_high = {}
for u in semantics.utterances:
    suspicions_low[u] = pragmatic_listener_1_low.get_suspicion(u)
    suspicions_high[u] = pragmatic_listener_1_high.get_suspicion(u)
    
utterance_order = [
    ("none", "ineffective"),
    ("none", "effective"),
    ("some", "ineffective"),
    ("some", "effective"),
    ("most", "ineffective"),
    ("most", "effective"),
    ("all", "ineffective"),
    ("all", "effective"),
]

utterances = [f"{q2} {pred}" for (q2, pred) in utterance_order]
probs_low = [suspicions_low.get((q2, pred), 0.0) for (q2, pred) in utterance_order]
probs_high = [suspicions_high.get((q2, pred), 0.0) for (q2, pred) in utterance_order]


title = "Suspicions of Utterances at round 0 (uniform prior)"
# numeric positions for y axis
y = np.arange(len(utterances))
height = 0.35  # thickness of each bar

plt.figure(figsize=(9, 6))

bars_low  = plt.barh(y + height/2, probs_low,  height, color="lightblue", label=r"$\alpha = $" + str(alpha_low))
bars_high = plt.barh(y - height/2, probs_high, height, color="steelblue", label=r"$\alpha = $" + str(alpha_high))

plt.xlabel("Suspicion")
plt.yticks(y, utterances)
plt.xlim(0, 1)
if alpha > 0:
    plt.title(f"{title}")
else:
    plt.title(title)

plt.legend()

# Add text labels
for bars, probs in [(bars_high, probs_high), (bars_low, probs_low)]:
    for bar, prob in zip(bars, probs):
        if prob > 0.01:
            plt.text(bar.get_width() + 0.01,
                     bar.get_y() + bar.get_height()/2,
                     f"{prob:.2f}",
                     va="center", fontsize=9)


In [ ]:
import matplotlib.pyplot as plt

# Convert your dictionaries into something plottable
def plot_rates(rates, title, ylabel):
    plt.figure(figsize=(8, 6))
    
    for threshold in thresholds:
        y_vals = [rates[(threshold, alpha)] for alpha in alphas]
        plt.plot(alphas, y_vals, marker='o', label=f"k={threshold}")
    
    plt.xlabel("alpha")
    plt.ylabel(ylabel)
    plt.title(title)
        # Move legend outside
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
    plt.grid(True)
    plt.show()
print(tprs[(0.7, 10.0)])
#print(tprs[(0.2, 40.0)])
# Plot TPR
plot_rates(tprs, r"True Positive Rate vs Alpha ($\theta = $" + str(theta) + ")" + " (Pers Ratio = " + str(pers_ratio) + ")", "TPR")

# Plot FPR
plot_rates(fprs, r"False Positive Rate vs Alpha ($\theta = $" + str(theta) + ")", "FPR")

In [ ]:
import matplotlib.pyplot as plt

# Convert your dictionaries into something plottable
def plot_rates(rates, title, ylabel):
    plt.figure(figsize=(8, 6))
    
    for threshold in thresholds:
        y_vals = [rates[(threshold, alpha)] for alpha in alphas]
        plt.plot(alphas, y_vals, marker='o', label=f"t={threshold}")
    
    plt.xlabel("alpha")
    plt.ylabel(ylabel)
    plt.title(title)
        # Move legend outside
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
    plt.grid(True)
    plt.show()

#print(tprs[(0.2, 40.0)])
# Plot TPR
plot_rates(tprs, "True Positive Rate vs Alpha", "TPR")

# Plot FPR
plot_rates(fprs, "False Positive Rate vs Alpha", "FPR")

In [ ]:
import seaborn as sns

def plot_heatmap(rates, title, ylabel):
    data = np.array([[rates[(t, a)] for a in alphas] for t in thresholds])
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(data, xticklabels=alphas, yticklabels=thresholds, annot=True)
    plt.xlabel("alpha")
    plt.ylabel("threshold")
    plt.title(title)
    plt.show()

plot_heatmap(tprs, "TPR heatmap", "threshold")
plot_heatmap(fprs, "FPR heatmap", "threshold")

In [ ]:
from itertools import product
import pickle
theta = 0.9
thetas = [0.1 * i for i in range(0, 11)]
psis = ["inf", "high", "low"]
world_parameters = {"n": 5, "m": 7}
alpha = 3.0
rounds = 50
#speaker_type = "low"
#listener_type = "vig"

world = World(theta, world_parameters, get_obs_prob, generate_all_observations)
quantifiers = ["none", "some", "most", "all"]
predicates = ["effective", "ineffective"]
if world_parameters["n"] > 1:
    utterances = list(product(quantifiers, quantifiers, predicates))
else:
    utterances = list(product(quantifiers, predicates))

semantics = Semantics(utterances, utterance_is_true)

exp_total = 10
for i in range(5, 10):
    for listener_type in ["vig"]:
        for speaker_type in ["low"]:
            for theta in [0.8]:
                world = World(theta, world_parameters, get_obs_prob, generate_all_observations)
                #starting experiment
                print(f"Starting experiment {i+1}/{exp_total}, speaker: {speaker_type}, listener: {listener_type}, theta: {theta}")
                logger, agents = run_game(thetas = thetas, psis = psis, semantics = semantics, world = world, alpha=alpha, rounds=rounds, speaker_type=speaker_type, listener_type=listener_type, verbose=False)
                filename = f"exp_{i+1}_s-{speaker_type}_l-{listener_type}_t-{theta}"
                with open(filename + "_logger.pkl", 'wb') as f:
                    pickle.dump(logger, f)
                with open(filename + "_agents.pkl", 'wb') as f:
                    pickle.dump(agents, f)
                print(f"Finished experiment {i+1}/{exp_total}, speaker: {speaker_type}, listener: {listener_type}, theta: {theta}")

In [ ]:
import pickle
with open("ag.pkl", "wb") as f:
    pickle.dump(agents_4, f)

# Load
with open("ag.pkl", "rb") as f:
    agents_reloaded = pickle.load(f)
    
agents_reloaded['pragmatic_listener_1'].hist[0].marginal(1)

In [ ]:
psis = {"inf": [], "high": [], "low": []}
for belief in agents_4["pragmatic_listener_1"].hist[0:40]:
    for (psi, prob) in belief.marginal(1).items():
        psis[psi].append(prob)

import matplotlib.pyplot as plt
for psi, values in psis.items():
    #make inf green, high blue, low red
    if psi == "inf":
        plt.plot(values, label=psi, color="green")
    elif psi == "high":
        plt.plot(values, label=psi, color="blue")
    elif psi == "low":
        plt.plot(values, label=psi, color="red")
plt.ylim(0, 1.1)
#add grid
plt.grid(True)
plt.xlabel("Round")
plt.ylabel("Probability")
plt.title("Belief Updates Over Time")
plt.legend()
plt.show()

In [ ]:
import numpy as np

def summarize_beliefs(belief_hist):
    thetas = np.array(sorted(belief_hist[0].keys()))
    means, ses = [], []
    for dist in belief_hist:
        ps = np.array([dist[th] for th in thetas])
        mean = np.sum(thetas * ps)
        var = np.sum((thetas - mean)**2 * ps)
        std = np.sqrt(var)
        # SE = std / sqrt(N); here N=1 distribution per round, so just use std as uncertainty
        se = std
        means.append(mean)
        ses.append(se)
    return np.arange(1, len(belief_hist)+1), np.array(means), np.array(ses)

theta_beliefs = []
for belief in agents_3["pragmatic_listener_1"].hist:
    theta_beliefs.append(belief.marginal(0))

rounds, means, ses = summarize_beliefs(theta_beliefs)

plt.figure(figsize=(8,5))
plt.plot(rounds, means, label="mean θ", color="blue")
plt.fill_between(rounds, means - ses, means + ses, color="blue", alpha=0.2, label="± std")

plt.xlabel("Round")
plt.ylabel("θ belief")
plt.title("Listener Belief Over Rounds")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
logger.to_json(filename="experiment_1_log.json", serializable=True)

In [ ]:
import json

# Load the file
with open("experiment_1_log.json", "r") as f:
    data = json.load(f)

# Fetch belief history of one agent
ps2_hist = data["agent_histories"]["pragmatic_speaker_2"]

print("Belief history of pragmatic_speaker_2:", ps2_hist[0])


In [ ]:
from itertools import product
theta = 0.3
thetas = [0.1 * i for i in range(0, 11)]
psis = ["inf", "high", "low"]
world_parameters = {"n": 5, "m": 7}
alpha = 3.0
world = World(theta, world_parameters, get_obs_prob, generate_all_observations)
quantifiers = ["none", "some", "most", "all"]
predicates = ["effective", "ineffective"]
if world_parameters["n"] > 1:
    utterances = list(product(quantifiers, quantifiers, predicates))
else:
    utterances = list(product(quantifiers, predicates))

semantics = Semantics(utterances, utterance_is_true)
speaker_low, listener_low = Game(thetas, psis, semantics, world, alpha=alpha, rounds=10)


In [ ]:
#plot informativity probability
import matplotlib.pyplot as plt
hist = listener.hist
y = []
for belief in hist:
    marg = belief.marginal(1)["inf"]
    y.append(marg)
plt.plot(y, marker='o')

In [ ]:
#plot informativity probability
import matplotlib.pyplot as plt
hist = listener_low.hist
y = []
for belief in hist:
    marg = belief.marginal(1)["low"]
    y.append(marg)
plt.plot(y, marker='o')

In [ ]:

theta = 0.3
world_parameters = {"n": 5, "m": 7}
alpha = 3.0
world = World(theta, world_parameters, get_obs_prob, generate_all_observations)


quantifiers = ["none", "some", "most", "all"]
predicates = ["effective", "ineffective"]
if world_parameters["n"] > 1:
    utterances = list(product(quantifiers, quantifiers, predicates))
else:
    utterances = list(product(quantifiers, predicates))

semantics = Semantics(utterances, utterance_is_true)
thetas = [0.1 * i for i in range(0, 11)]
literal_speaker = Speaker0(thetas, semantics=semantics, world=world)
literal_listener = Listener0(thetas, literal_speaker, world=world, semantics=semantics)
pragmatic_speaker = Speaker1(thetas, literal_listener, semantics=semantics, world=world, alpha=alpha, psi="inf")
pragmatic_listener = Listener1(thetas, ["inf", "high", "low"], pragmatic_speaker, semantics=semantics, world=world, alpha=alpha)
#pragmatic_speaker.dist_over_utterances_obs((0, 4, 0, 0, 0, 0, 1, 0), "inf")
# pragmatic_speaker.dist_over_utterances_theta(0.3, "inf")
pragmatic_speaker2 = Speaker2(thetas, pragmatic_listener, semantics=semantics, world=world, alpha=alpha, psi="inf")
pragmatic_speaker2.dist_over_utterances_obs((0, 4, 0, 0, 0, 0, 1, 0), "inf")


In [ ]:
from itertools import product
from functools import lru_cache
quantifiers = ["none", "some", "most", "all"]
predicates = ["effective", "ineffective"]

def generate_all_observations(n, m):
    """
    Generate all possible observation histograms for n patients and m sessions.
    Each histogram is a tuple of length m+1, summing to n.
    """
    observations = []

    def helper(current, depth, remaining):
        if depth == m:
            current.append(remaining)
            observations.append(tuple(current))
            current.pop()
            return
        for i in range(remaining + 1):
            current.append(i)
            helper(current, depth + 1, remaining - i)
            current.pop()

    helper([], 0, n)
    return observations

def literal_speaker_utterance_obs(obs, quantifiers=quantifiers, predicates=predicates):
    """Return uniform distribution over all true utterances"""
    if sum(obs) > 1:
        utterances = list(product(quantifiers, quantifiers, predicates))
    else:
        utterances = list(product(quantifiers, predicates))
    true_utterances = [u for u in utterances if utterance_is_true(u, obs)]
    if not true_utterances:
        return {}
    p = 1.0 / len(true_utterances)
    result = {}
    for u in utterances:
        if u in true_utterances:
            result[u] = p
        else:
            result[u] = 0.0
    return result

@lru_cache(maxsize=None)
def literal_speaker_utterance_theta(theta, n, m, quantifiers=quantifiers, predicates=predicates, N = 1000):
    """Estimate literal speaker utterance distribution given theta via sampling"""
    if n > 1:
        utterances = list(product(quantifiers, quantifiers, predicates))
    else:
        utterances = list(product(quantifiers, predicates))
        
    probs = {u: 0.0 for u in utterances}
    observations = generate_all_observations(n, m)
    for obs in observations:
        obs_prob = get_obs_prob(obs, theta)
        for utt, prob in literal_speaker_utterance_obs(obs).items():
            probs[utt] += prob * obs_prob
    return probs

def literal_listener_theta_utterance(utterance, n = 10, m = 20, thetas=thetas, N = 1000):
    probs = {}
    for theta in thetas:
        utterance_probs = literal_speaker_utterance_theta(theta, n=n, m=m)
        probs[theta] = utterance_probs.get(tuple(utterance), 0.0) / len(thetas)
    total = sum(probs.values())
    return [p / total for p in probs.values()]


@lru_cache(maxsize=None)
def get_literal_listener_utt(n, m, thetas_tuple, quantifiers_tuple):
    if n > 1:
        utterances = list(product(quantifiers_tuple, quantifiers_tuple, ["effective", "ineffective"]))
    else:
        utterances = list(product(quantifiers_tuple, ["effective", "ineffective"]))

    all_obs = generate_all_observations(n, m)
    utt_priors = {}
    for utt in utterances:
        total = 0.0
        for obs_case in all_obs:
            literal_speaker_utterance_obscase_val = literal_speaker_utterance_obs(obs_case, quantifiers_tuple, ["effective", "ineffective"])
            for theta_case in thetas_tuple:
                obs_prior = get_obs_prob(obs_case, theta_case) 
                total += (
                    literal_speaker_utterance_obscase_val[utt]
                    * obs_prior
                    * (1 / len(thetas_tuple))
                )
        utt_priors[utt] = total
    return utt_priors

def informativeness_all_utterances(obs, thetas, quantifiers=["none", "some", "most", "all"], N=1000):
    n = sum(obs)
    m = len(obs) - 1

    # ---- cache for literal_listener_utt ----


    # ---- normal part depending on obs ----
    if n > 1:
        utterances = list(product(quantifiers, quantifiers, ["effective", "ineffective"]))
    else:
        utterances = list(product(quantifiers, ["effective", "ineffective"]))

    result = {}
    total_success = sum(obs[i] * i for i in range(len(obs)))
    literal_speaker_utterance_obs_val = literal_speaker_utterance_obs(obs, quantifiers, ["effective", "ineffective"])

    literal_listener_obs = 0
    for theta in thetas:
        literal_listener_obs += get_obs_prob(obs, theta) * (1 / len(thetas))

    # get precomputed priors (cached internally)
    utt_priors = get_literal_listener_utt(n, m, tuple(thetas), tuple(quantifiers))

    for utt in utterances:
        result[utt] = (
            literal_speaker_utterance_obs_val[utt]
            * literal_listener_obs
            / utt_priors[utt]
        )
    return result


def persuasiveness_all_utterances(pers, n = 10, m = 20, thetas=thetas, quantifiers=["none", "some", "most", "all"], predicates=["effective", "ineffective"], N = 1000):
    if n > 1:
        utterances = list(product(quantifiers, quantifiers, predicates))
    else:
        utterances = list(product(quantifiers, predicates))
        
    result = {u: 0.0 for u in utterances}

    for utt in utterances:
        if pers == "inf":
            result[utt] = 1
        elif pers == "high":
            theta_dist = literal_listener_theta_utterance(utt, n=n, m=m, thetas=thetas, N=N)
            for i in range(len(theta_dist)):
                result[utt] += theta_dist[i] * thetas[i]
        elif pers == "low":
            theta_dist = literal_listener_theta_utterance(utt, n=n, m=m, thetas=thetas, N=N)
            for i in range(len(theta_dist)):
                result[utt] += theta_dist[i] * thetas[i]
            result[utt] = 1 - result[utt]
    return result

def pragmatic_speaker_utt_obs(obs, pers, alpha=1.0, thetas=np.linspace(0.1, 1.0, 10), N=1000):
    n = sum(obs)
    m = len(obs) - 1
    quantifiers = ["none", "some", "most", "all"]
    predicates = ["effective", "ineffective"]

    # Get utterance space
    if n > 1:
        utterances = list(product(quantifiers, quantifiers, predicates))
    else:
        utterances = list(product(quantifiers, predicates))

    # Compute informativeness and persuasiveness
    informativity_dict = informativeness_all_utterances(obs, thetas, quantifiers=quantifiers, N=N)
    persuasiveness_dict = persuasiveness_all_utterances(pers, n=n, m=m, thetas=thetas, quantifiers=quantifiers, predicates=predicates, N=N)

    # Compute softmax weights
    scores = []
    if pers == "inf":
        beta = 1.0
    else:
        beta = 0.0

    for utt in utterances:
        info = informativity_dict.get(utt, 0.0)
        pers_val = persuasiveness_dict.get(utt, 0.0)

        if info > 0:
            score = (info ** (alpha * beta)) * (pers_val ** (alpha * (1 - beta)))
        else:
            score = 0.0
        scores.append(score)

    scores = np.array(scores)
    probs = scores / np.sum(scores) if np.sum(scores) > 0 else np.ones_like(scores) / len(scores)

    return {utt: p for utt, p in zip(utterances, probs)}

def pragmatic_speaker_utt_theta(theta, pers, alpha=1.0, thetas=np.linspace(0.1, 1.0, 10), n=1, m=7):
    quantifiers = ["none", "some", "most", "all"]
    predicates = ["effective", "ineffective"]

    # Get utterance space
    if n > 1:
        utterances = list(product(quantifiers, quantifiers, predicates))
    else:
        utterances = list(product(quantifiers, predicates))
    result = {u: 0.0 for u in utterances}
    all_observations = generate_all_observations(n = n, m = m)
    for obs in all_observations:
        mid_results = pragmatic_speaker_utt_obs(obs, pers=pers, alpha=alpha, thetas=thetas)
        total_success = sum([i * obs[i] for i in range(len(obs))])
        obs_prob = get_obs_prob(obs, theta)
        for utt in utterances:      
            result[utt] += mid_results[utt] * obs_prob
    return result

def pragmatic_listener_theta_psi_utt(utterance, thetas, psis, omega, n = 1, m = 7, alpha = 3.0):
    joint = {}
    all_observations = generate_all_observations(n = n, m = m)
    for theta, psi in product(thetas, psis):
        middle = 0

        for obs in all_observations:
            total_success = sum([i * obs[i] for i in range(len(obs))])
            obs_prob = math.comb(n * m, total_success) * (theta ** total_success) * (1 - theta) ** (n * m - total_success)
            middle += pragmatic_speaker_utt_obs(obs, pers=psi, alpha=alpha, thetas=thetas)[utterance] * obs_prob
        if omega == "strat":
            joint[psi, theta] = middle * (1 / len(thetas)) * (1 / len(psis))
        elif omega == "coop":
            if psi == "inf":
                joint[psi, theta] = middle * (1 / len(thetas))
            else:
                joint[psi, theta] = 0

    total = sum(joint.values())
    return {k: v / total for k, v in joint.items()}

def pragmatic_listener_theta_utt(joint):
    theta_probs = {}
    for (psi, theta), prob in joint.items():
        theta_probs[theta] = theta_probs.get(theta, 0.0) + prob
    return theta_probs

def pragmatic_listener_psi_utt(joint):
    psi_probs = {}
    for (psi, theta), prob in joint.items():
        psi_probs[psi] = psi_probs.get(psi, 0.0) + prob
    return psi_probs

In [ ]:
pragmatic_speaker_utt_obs((0, 1, 0, 0, 0, 0, 1, 0), pers="inf", alpha=3.0, thetas=thetas)

In [ ]:
n = 5
m = 7
speaker_dist = pragmatic_speaker_utt_theta(theta = 0.90, pers="high", alpha=3.0, n=n, m=m)
print("total: ", sum(speaker_dist.values()))
# Print top utterances
for utt, p in sorted(speaker_dist.items(), key=lambda x: -x[1]):
    print(f"{utt}: {p:.3f}")

In [ ]:
obs = (0, 1, 1, 1, 0, 0, 0, 2)
speaker_dist = pragmatic_speaker_utt_obs(obs, pers="inf", alpha=3.0)
print("total: ", sum(speaker_dist.values()))
# Print top utterances
for utt, p in sorted(speaker_dist.items(), key=lambda x: -x[1]):
    print(f"{utt}: {p:.3f}")

In [ ]:
import matplotlib.pyplot as plt

def plot_literal_listener_theta_posterior(probs, thetas, utterance):
    """Plot posterior distribution over θ given an utterance."""
    plt.figure(figsize=(8, 4))
    plt.plot(thetas, probs, marker='o', color='blue', linewidth=2)
    if len(utterance) == 3:
        plt.title(f"Literal Listener Posterior: P(θ | {utterance[0]}-{utterance[1]} {utterance[2]})")
    else:
        plt.title(f"Literal Listener Posterior: P(θ | {utterance[0]} {utterance[1]})")
    plt.xlabel("θ (Effectiveness rate)")
    plt.ylabel("P(θ | utterance)")
    plt.xticks(thetas)
    plt.ylim(0, max(probs) * 1.1)  # Adjust y-axis limit for better visibility
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
def suspicion_proper(utt, thetas, n = 1, m = 7, alpha = 3.0):

    sus = 0
    all_obs = generate_all_observations(n = n, m = m)
    for obs in all_obs:
        state_priors = informativeness_all_utterances(obs=obs, thetas=thetas)
        speaker_result = pragmatic_speaker_utt_obs(obs, pers="inf", alpha=alpha, thetas=thetas)
        for u, prob in speaker_result.items():
            if np.isclose(prob, speaker_result[utt], rtol=1e-9, atol=1e-12):
                continue
            elif prob > speaker_result[utt]:
                sus += state_priors[utt] * prob
    return sus


In [ ]:
utt = ("some", "effective")
thetas = np.linspace(0.0, 1.0, 11)
n = 1
m = 7
alpha = 3.0
sus = suspicion_proper(utt = utt, thetas = thetas, n=n, m=m, alpha=alpha)

print(f"Suspicion of utterance {utt}: {sus:.4f}")

In [ ]:
utterance = ("some","some", "effective")
thetas = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
psis = ["high", "low", "inf"]
n = 5
m = 7
alpha = 3.0

joint = pragmatic_listener_theta_psi_utt(utterance=utterance, thetas=thetas, psis=psis, omega = "strat", n=n, m=m, alpha=alpha)
theta_dist = [p for p in pragmatic_listener_theta_utt(joint).values()]
psis_dist = pragmatic_listener_psi_utt(joint)
print(psis_dist)
high_psi = 0
low_psi = 0
inf_psi = 0
for (psi, theta), prob in joint.items():
    if psi == "high":
        high_psi += theta * prob
    elif psi == "low":
        low_psi += theta * prob 
    elif psi == "inf":
        inf_psi += theta * prob
print(f"High Persuasiveness: {high_psi}, Low Persuasiveness: {low_psi}, Informative: {inf_psi}")
plot_literal_listener_theta_posterior(theta_dist, thetas, utterance)

theta_dist = literal_listener_theta_utterance(utterance, n=n, m=m, thetas=thetas)
plot_literal_listener_theta_posterior(theta_dist, thetas, utterance)

joint = pragmatic_listener_theta_psi_utt(utterance=utterance, thetas=thetas, psis=psis, omega = "coop", n=n, m=m, alpha=alpha)
theta_dist = [p for p in pragmatic_listener_theta_utt(joint).values()]
plot_literal_listener_theta_posterior(theta_dist, thetas, utterance)

In [ ]:
print(literal_listener_theta_utterance(("some", "effective"), n=1, m=7, thetas=thetas, N=1000))
print(sum(literal_listener_theta_utterance(("some", "effective"), n=1, m=7, thetas=thetas, N=1000)) / len(thetas))

print(persuasiveness_all_utterances("low", n = 1, m = 7, thetas=thetas, quantifiers=["none", "some", "most", "all"], predicates=["effective", "ineffective"], N = 1000))

In [ ]:
obs = observations[4]  # Example observation
utt = ("most", "ineffective")
is_true = utterance_is_true(utt, obs)
print(f"Is the utterance {utt} true for the first observation?\n {obs} \n {is_true}")

In [ ]:
res = literal_speaker_utterance_theta(0.8, 1, 7)
sum_res = sum(res.values())
print(f"Sum of probabilities for utterances: {sum_res}")
print(res)

In [ ]:
res = literal_speaker_utterance_theta(0.8, 1, 7)
sum_res = sum(res.values())
print(f"Sum of probabilities for utterances: {sum_res}")
print(res)

In [ ]:
utterance = ("some", "effective")
observation = np.array([[0, 0, 0, 0, 0, 0, 0]])
dist = literal_speaker_utterance_obs(observation)
print(literal_listener_theta_utterance(utterance, n=1, m=7))
print(f"Utterance: {utterance}, Observation: {observation}, Distribution: {dist}")
utterance_is_true(utterance, observation)

In [ ]:

import matplotlib.pyplot as plt
import numpy as np

def plot_utterance_histogram(distribution):
    quantifiers = ["none", "some", "most", "all"]
    preds = ["effective", "ineffective"]

    # Create x-axis labels from quantifier pairs
    x_labels = [f"{q1}-{q2}" for q1 in quantifiers for q2 in quantifiers]

    # Create y-values for both predicates
    values = {pred: [] for pred in preds}
    for q1 in quantifiers:
        for q2 in quantifiers:
            for pred in preds:
                prob = distribution.get((q1, q2, pred), 0.0)
                values[pred].append(prob)

    x = np.arange(len(x_labels))
    width = 0.35  # width of each bar

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(x - width/2, values["effective"], width, label="Effective", color="#4caf50")
    ax.bar(x + width/2, values["ineffective"], width, label="Ineffective", color="#f44336")

    ax.set_ylabel("Probability")
    ax.set_xlabel("Utterance (q1-q2)")
    ax.set_title("Literal Speaker Utterance Distribution")
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, rotation=45, ha="right")
    ax.legend()
    ax.set_ylim(0, 1)

    plt.tight_layout()
    plt.show()



In [ ]:
utt = ("most", "effective")
probs = literal_listener_theta_utterance(utt,thetas=thetas, n=1, m=7)
plot_literal_listener_theta_posterior(probs, thetas, utt)
#plot_utterance_histogram(probs)

In [ ]:
probs = literal_listener_theta_utterance(("some", "effective"),thetas=thetas, n=1, m=7, N=10000)
plot_literal_listener_theta_posterior(probs, thetas, ("most", "effective"))
#plot_utterance_histogram(probs)

In [ ]:
def informativeness_speaker1(utterance, obs, thetas=thetas):
    if utterance_is_true(utterance, obs):
        theta_utt = literal_listener_theta_utterance(utterance, n=obs.shape[0], m=obs.shape[1], thetas=thetas)
        result = 0
        for theta in thetas:
            obs_theta = (theta ** np.sum(obs)) * ((1 - theta) ** (obs.size - np.sum(obs)))
            result += obs_theta * theta_utt[int(theta * 10 - 1)]
        return result
        
    else:
        return 0.0
    

In [ ]:
from itertools import product
import numpy as np

def informativeness_all_utterances(obs, thetas, quantifiers=["none", "some", "most", "all"], N=1000):
    if obs.shape[0] > 1:
        utterances = list(product(quantifiers, quantifiers, ["effective", "ineffective"]))
    else:
        utterances = list(product(quantifiers, ["effective", "ineffective"]))
    result = {}
    print(f"obs shape: {obs.shape}, utterances: {len(utterances)}")
    total_successes = np.sum(obs)
    total_trials = obs.size

    for utt in utterances:
        if utterance_is_true(utt, obs):
            print(utt)
            theta_utt = literal_listener_theta_utterance(
                utt, n=obs.shape[0], m=obs.shape[1], thetas=thetas, N=N
            )
            info = 0.0
            for i, theta in enumerate(thetas):
                p_obs_given_theta = math.comb(obs.size, total_successes)* (theta ** total_successes) * ((1 - theta) ** (total_trials - total_successes))
                info += p_obs_given_theta * theta_utt[i]
            result[utt] = info
        else:
            result[utt] = 0.0

    return result


def pragmatic_speaker1_softmax(informativity_dict, alpha=1.0):
    utterances = list(informativity_dict.keys())
    probs = []
    for utt, info in informativity_dict.items():
        if info > 0:
            probs.append(info ** alpha)
        else:
            probs.append(0.0)
    probs = np.array(probs)
    probs /= np.sum(probs)  # Normalize to ensure probabilities sum to 1
    
    
    # infos = np.array([informativity_dict[u] for u in utterances])

    # infos_nonzero = infos[infos > 0]
    # # Softmax with temperature (α): exponentiate scaled informativeness
    # scaled = alpha * infos
    # # scaled -= np.max(scaled)  # For numerical stability
    # exp_scaled = np.exp(scaled)
    # exp_scaled.map(lambda x: 0 if x == 1 else x)  # Avoid division by zero
    # probs = exp_scaled / np.sum(exp_scaled)

    return {u: p for u, p in zip(utterances, probs)}

In [ ]:
def plot_speaker_distribution(distribution, title="Speaker Distribution", alpha=1.0):
    """Plot speaker distribution over utterances (q2, predicate) in a fixed order."""
    utterance_order = [
        ("none", "ineffective"),
        ("none", "effective"),
        ("some", "ineffective"),
        ("some", "effective"),
        ("most", "ineffective"),
        ("most", "effective"),
        ("all", "ineffective"),
        ("all", "effective"),
    ]

    utterances = [f"{q2} {pred}" for (q2, pred) in utterance_order]
    probs = [distribution.get((q2, pred), 0.0) for (q2, pred) in utterance_order]

    plt.figure(figsize=(8, 5))
    bars = plt.barh(utterances, probs, color="steelblue")
    plt.xlabel("Probability")
    if alpha > 0:
        plt.title(f"{title} (α={alpha})")
    else:
        plt.title(title)
    plt.xlim(0, 1)

    # Add text labels
    for bar, prob in zip(bars, probs):
        if prob > 0.01:
            plt.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                     f"{prob:.2f}", va="center", fontsize=9)

    plt.gca().invert_yaxis()  # Keep top-down order
    plt.tight_layout()

In [ ]:
obs = (1, 0, 0, 0, 0, 0, 0, 0)
print(obs)
infos = informativeness_all_utterances(obs, thetas=np.linspace(0.1, 1.0, 10), N = 10000)
perss = persuasiveness_all_utterances("high", n = 1, m = 7, thetas=np.linspace(0.1, 1.0, 10), quantifiers=["none", "some", "most", "all"], predicates=["effective", "ineffective"], N = 1000)
speaker1_dist = pragmatic_speaker1_softmax(infos, perss, alpha=3.0, beta=0)
print(infos)
for u, p in sorted(speaker1_dist.items(), key=lambda x: -x[1]):
    if p > 0:
        print(f"{u}: {p}")

In [ ]:
cases = [(1, 0, 0, 0, 0, 0, 0, 0),
         (0, 0, 1, 0, 0, 0, 0, 0),
         (0, 0, 0, 0, 1, 0, 0, 0)]
for obs in cases:
    total_success = sum([obs[i] * i for i in range(len(obs))])
    infos = informativeness_all_utterances(obs, thetas=np.linspace(0.1, 1.0, 10), N = 10000)
    dist = pragmatic_speaker1_softmax(infos, alpha=3.0)
    plot_speaker_distribution(literal_speaker_utterance_obs(obs), title=f"Literal Speaker Distribution {total_success} / {len(obs) - 1}")
    plot_speaker_distribution(dist, title=f"Pragmatic Speaker Distribution {total_success} / {len(obs) - 1}", alpha=0)
    

In [ ]:
plot_speaker_distribution(speaker1_dist, title="Pragmatic Speaker 1 Distribution", alpha=3.0)

In [ ]:
print(obs)
for u, v in sorted(informativity_dict.items(), key=lambda x: -x[1]):
    if v > 0:
        print(f"{u}: {v}")


In [ ]:
obs = create_samples(0.8, 10, 5)
print("obs:\n", obs)

distribtuion = literal_speaker_utterance(obs)
def print_distribution(dist, threshold=0.0):
    print(f"{'Quantifier 1':<10} {'Quantifier 2':<10} {'Effectiveness':<12} {'Probability':<10}")
    print("-" * 50)
    
    for key, prob in sorted(dist.items(), key=lambda x: -x[1]):
        if prob >= threshold:
            q1, q2, eff = key
            print(f"{q1:<10} {q2:<10} {eff:<12} {prob:<10.3f}")
            
print_distribution(distribtuion, threshold=0.0)

In [ ]:
entropies = {}
utterances = list(product(quantifiers, predicates))
n = 1
m = 7
for utterance in utterances:
    dist = literal_listener_theta_utterance(utterance, n=n, m=m)
    entropy = -sum(p * np.log2(p) for p in dist if p > 0)
    entropies[utterance] = entropy

preciseness1 = {u: 1 / np.exp(ent) for u, ent in entropies.items()}
max_entropy = max(entropies.values())
min_entropy = min(entropies.values())
preciseness2 = {u: 1 - (ent - min_entropy) / (max_entropy - min_entropy) for u, ent in entropies.items()}


#plot entropies
plt.figure(figsize=(10, 5))
plt.bar(range(len(entropies)), list(entropies.values()), align='center')
plt.xticks(range(len(entropies)), [f"{q2} {pred}" for (q2, pred) in entropies.keys()], rotation=45)
plt.xlabel("Utterance")
plt.ylabel("Entropy")
plt.title("Entropy of Utterances")
plt.tight_layout()

#plot precisions
plt.figure(figsize=(10, 5))
plt.bar(range(len(preciseness1)), list(preciseness1.values()), align='center', label='Preciseness 1')
plt.bar(range(len(preciseness2)), list(preciseness2.values()), align='center', label='Preciseness 2', alpha=0.7)
plt.xticks(range(len(preciseness1)), [f"{q2} {pred}" for (q2, pred) in preciseness1.keys()], rotation=45)
plt.xlabel("Utterance")
plt.ylabel("Preciseness")
plt.legend()

In [ ]:
preciseness = {}
for interval_range in range(0, 101):
    